# Accesibilidad QroBus hacia la estación Corregidora

Este notebook identifica:

1. **Paradas de origen** desde las que se puede llegar a la estación Corregidora en 30 minutos o menos.
2. **Itinerarios de rutas** que permiten llegar dentro del umbral, tanto directos como con transbordos.
3. Un mapa principal con capas consistentes y otro mapa con todas las paradas de 30 minutos que no requieren cruzar las vías.
4. La demanda potencial cubierta, usando la malla geoestadística del INEGI (celdas de unos 500 m, proporcionada por el Instituto de Movilidad) y buffers configurables alrededor de las paradas conectadas.

El cálculo siempre se realiza **hacia Corregidora**. Los datos fuente están en `data/gtfs/`, los resultados en `data/processed/` y los HTML en `maps/`.


## 1. Metodología y alcance

Se construye un grafo dirigido con cada par consecutivo de paradas de los viajes GTFS. El tiempo de cada segmento es la mediana de los tiempos programados para la combinación `origen-destino-ruta`.

Para calcular caminos con transbordos se aplica Dijkstra sobre estados `(parada, ruta)`. Cambiar de ruta agrega una penalización configurable mediante `PENALIZACION_TRANSBORDO_MIN`.

> La penalización representa espera y caminata de conexión de forma aproximada. Un cálculo exacto necesita horarios por fecha, tiempos de espera y, preferentemente, GTFS-Realtime. El notebook no presenta esta estimación como telemetría real.

Si existe `data/processed/tiempos_corregidos_google.csv`, se suma el atraso promedio disponible al tiempo de red. Si no existe, el análisis funciona únicamente con GTFS y lo indica explícitamente.


## 2. Correcciones realizadas durante QA

Se eliminaron:

- tres mapas parcialmente duplicados;
- análisis en sentido contrario, desde Corregidora hacia otras paradas;
- dependencias no necesarias como GeoPandas, Shapely, SciPy y Matplotlib;
- rutas geométricas construidas con un viaje representativo que podía pertenecer a la dirección incorrecta;
- bloques repetidos y capturas silenciosas de excepciones.

También se corrigió el modelo de transbordos: cambiar de ruta ya no tiene costo cero.


## 3. Configuración

Las carpetas, las coordenadas de la estación y los parámetros de `.env` se cargan desde `src/scripts/config.py`, compartido por los tres notebooks. Los parámetros principales son:

```dotenv
UMBRAL_ANALISIS_MIN=30
PENALIZACION_TRANSBORDO_MIN=5
```

`02_RutasQroBus.ipynb` no llama a Google Maps. Solo consume la corrección ya generada por `01_PromedioRutas.ipynb`, cuando está disponible.


In [1]:
from pathlib import Path
import colorsys
import hashlib
import html
import sys

import folium
import numpy as np
import pandas as pd
from IPython.display import display

# Los módulos compartidos viven en src/scripts/; funciona al ejecutar desde
# src/notebooks/, desde src/ o desde la raíz del repositorio.
SCRIPTS_DIR = next(
    base / "scripts"
    for base in [Path.cwd().parent, Path.cwd(), Path.cwd() / "src"]
    if (base / "scripts" / "config.py").exists()
)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import (
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
    ENV_CARGADO,
    GTFS_DIR,
    MALLA_INEGI_DIR,
    MAPS_DIR,
    MAX_EDAD_ETA_HORAS,
    MIN_OBSERVACIONES_HISTORICAS,
    PENALIZACION_TRANSBORDO_MIN,
    PROCESSED_DIR,
    RADIO_DEMANDA_M,
    UMBRAL_ANALISIS_MIN,
)
from geo import (
    TRAZO_LINEA_RECTA,
    VIA_FERREA_CORREGIDORA_LATLON,
    cargar_trazos,
    guardar_capa_qgis,
    haversine_m,
    tabla_con_coordenadas,
)
from gtfs_network import cargar_rutas_base, compactar_rutas
from malla_inegi import cargar_malla, estimar_demanda_malla

print(f"GTFS: {GTFS_DIR}")
print(f"Procesados: {PROCESSED_DIR}")
print(f"Mapas: {MAPS_DIR}")
print(f".env cargado: {ENV_CARGADO}")
print(f"Umbral: {UMBRAL_ANALISIS_MIN:g} min")
print(f"Penalización por transbordo: {PENALIZACION_TRANSBORDO_MIN:g} min")
print(f"Vigencia máxima de ETA Google: {MAX_EDAD_ETA_HORAS:g} h")
print(f"Muestra histórica mínima: {MIN_OBSERVACIONES_HISTORICAS} observaciones")


GTFS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/gtfs
Procesados: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed
Mapas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps
.env cargado: True
Umbral: 30 min
Penalización por transbordo: 5 min
Vigencia máxima de ETA Google: 24 h
Muestra histórica mínima: 5 observaciones


### 3.1 Funciones de visualización

Estas funciones viven en el notebook porque solo se usan aquí. Generan colores estables por ruta, extraen únicamente segmentos conectados, les asignan su trazo sobre la red vial y construyen las leyendas de los mapas.


In [2]:
RUTAS_CAMINATA = {"CAMINATA", "CAMINATA_FINAL"}


def color_por_ruta(route_id):
    """Genera un color hexadecimal estable y reproducible por route_id."""
    texto = str(route_id)
    if texto in RUTAS_CAMINATA:
        return "#6b7280"
    digest = int(hashlib.sha1(texto.encode("utf-8")).hexdigest()[:8], 16)
    tono = (digest % 360) / 360
    rojo, verde, azul = colorsys.hsv_to_rgb(tono, 0.72, 0.82)
    return f"#{round(rojo * 255):02x}{round(verde * 255):02x}{round(azul * 255):02x}"


def etiqueta_ruta(route_id, route_name_map):
    """Traduce un identificador interno a la etiqueta pública de la ruta."""
    route_id_texto = str(route_id)
    if route_id_texto in RUTAS_CAMINATA:
        return "Caminata final"
    etiqueta = route_name_map.get(route_id_texto)
    return str(etiqueta) if pd.notna(etiqueta) else route_id_texto


def ruta_inicial(rutas_por_segmento):
    """Obtiene la ruta del primer salto de un camino."""
    if not isinstance(rutas_por_segmento, (list, tuple)) or not rutas_por_segmento:
        return None
    return str(rutas_por_segmento[0])


def distancia_camino_km(camino_stop_ids, trazos):
    """Suma la distancia por calles de un camino; NaN si falta algún tramo."""
    paradas = [str(stop_id) for stop_id in camino_stop_ids]
    pares = list(zip(paradas, paradas[1:]))
    if any(par not in trazos for par in pares):
        return np.nan
    return sum(trazos[par]["distancia_red_m"] for par in pares) / 1000


def tiempos_historicos_por_parada(paradas):
    """Indexa por stop_id el P50/P80 histórico hacia Corregidora de cada parada."""
    columnas = ["stop_name", "p50_historico_min", "p80_historico_min", "observaciones"]
    return (
        paradas.assign(stop_id=paradas["stop_id"].astype(str))
        .set_index("stop_id")[columnas]
        .to_dict("index")
    )


def extraer_segmentos_conectados(
    caminos, coords_por_stop, route_name_map, trazos, tiempos_por_stop=None
):
    """Extrae los segmentos reales de los caminos y elimina duplicados exactos.

    Cada segmento lleva su trazo sobre la red vial; si no existe, se dibuja
    la línea recta entre las dos paradas y queda marcado como tal. Con
    `tiempos_por_stop` también lleva el P50/P80 histórico hacia Corregidora de
    su parada de origen, siempre que la muestra alcance el mínimo configurado.
    """
    tiempos_por_stop = tiempos_por_stop or {}
    registros = {}
    for fila in caminos.itertuples(index=False):
        paradas = [str(stop_id) for stop_id in fila.camino_stop_ids]
        rutas = [str(route_id) for route_id in fila.rutas_por_segmento]
        if len(paradas) != len(rutas) + 1:
            raise ValueError(
                f"Camino inconsistente para {fila.stop_id}: "
                f"{len(paradas)} paradas y {len(rutas)} segmentos."
            )

        for stop_a, stop_b, route_id in zip(paradas, paradas[1:], rutas):
            coord_a = coords_por_stop.get(stop_a)
            coord_b = coords_por_stop.get(stop_b)
            if coord_a is None or coord_b is None:
                continue
            clave = (route_id, stop_a, stop_b)
            recta = [
                [float(coord_a["stop_lat"]), float(coord_a["stop_lon"])],
                [float(coord_b["stop_lat"]), float(coord_b["stop_lon"])],
            ]
            trazo = trazos.get((stop_a, stop_b))
            tiempos = tiempos_por_stop.get(stop_a, {})
            con_muestra = (
                tiempos.get("observaciones", 0) >= MIN_OBSERVACIONES_HISTORICAS
            )
            registros.setdefault(
                clave,
                {
                    "route_id": route_id,
                    "route_label": etiqueta_ruta(route_id, route_name_map),
                    "stop_id_a": stop_a,
                    "stop_id_b": stop_b,
                    "lat_a": float(coord_a["stop_lat"]),
                    "lon_a": float(coord_a["stop_lon"]),
                    "lat_b": float(coord_b["stop_lat"]),
                    "lon_b": float(coord_b["stop_lon"]),
                    "coordenadas": trazo["coordenadas"] if trazo else recta,
                    "distancia_m": trazo["distancia_red_m"] if trazo else np.nan,
                    "metodo_trazo": (
                        trazo["metodo_trazo"] if trazo else TRAZO_LINEA_RECTA
                    ),
                    "stop_name_a": tiempos.get("stop_name"),
                    "p50_min": (
                        tiempos["p50_historico_min"] if con_muestra else np.nan
                    ),
                    "p80_min": (
                        tiempos["p80_historico_min"] if con_muestra else np.nan
                    ),
                    "observaciones": tiempos.get("observaciones", 0),
                },
            )

    columnas = [
        "route_id", "route_label", "stop_id_a", "stop_id_b",
        "lat_a", "lon_a", "lat_b", "lon_b",
        "coordenadas", "distancia_m", "metodo_trazo",
        "stop_name_a", "p50_min", "p80_min", "observaciones",
    ]
    return pd.DataFrame(registros.values(), columns=columnas)


def separar_origenes_conectados(origenes, segmentos):
    """Separa paradas dibujables de las que no tienen un tramo saliente."""
    stop_ids_con_salida = set(segmentos["stop_id_a"].astype(str))
    mascara = origenes["stop_id"].astype(str).isin(stop_ids_con_salida)
    return origenes.loc[mascara].copy(), origenes.loc[~mascara].copy()


def agregar_segmentos_rutas(capa, segmentos, opacity=0.72):
    """Dibuja cada segmento sobre las calles con el color estable de su ruta."""
    for segmento in segmentos.itertuples(index=False):
        es_caminata = str(segmento.route_id) in RUTAS_CAMINATA
        etiqueta = f"Ruta: {segmento.route_label}"
        if pd.notna(segmento.distancia_m):
            etiqueta += f" | {segmento.distancia_m:,.0f} m"
        if segmento.metodo_trazo == TRAZO_LINEA_RECTA:
            etiqueta += " (línea recta: sin trazo vial)"
        if pd.notna(segmento.p50_min):
            etiqueta += (
                f" | Desde {segmento.stop_name_a} a Corregidora: "
                f"P50 {segmento.p50_min:.1f} min · P80 {segmento.p80_min:.1f} min "
                f"(n={segmento.observaciones})"
            )
        folium.PolyLine(
            segmento.coordenadas,
            color=color_por_ruta(segmento.route_id),
            weight=4 if not es_caminata else 3,
            opacity=opacity,
            dash_array="7 7" if es_caminata else None,
            tooltip=etiqueta,
        ).add_to(capa)


def agregar_leyenda_rutas(mapa, segmentos, titulo="Rutas"):
    """Agrega una leyenda desplazable con el rango P50/P80 de cada ruta.

    El rango de una ruta resume los tiempos históricos hacia Corregidora de las
    paradas donde inician sus segmentos dibujados.
    """
    rutas = (
        segmentos.drop_duplicates(["route_id", "stop_id_a"])
        .groupby(["route_id", "route_label"], as_index=False)
        .agg(
            p50_minimo=("p50_min", "min"),
            p50_maximo=("p50_min", "max"),
            p80_minimo=("p80_min", "min"),
            p80_maximo=("p80_min", "max"),
        )
        .sort_values(["route_label", "route_id"])
    )
    elementos = []
    for fila in rutas.itertuples(index=False):
        color = color_por_ruta(fila.route_id)
        tiempos = (
            f" — P50 {fila.p50_minimo:.0f}–{fila.p50_maximo:.0f} · "
            f"P80 {fila.p80_minimo:.0f}–{fila.p80_maximo:.0f} min"
            if pd.notna(fila.p50_minimo)
            else ""
        )
        elementos.append(
            "<div style='white-space:nowrap;margin:1px 0'>"
            f"<span style='display:inline-block;width:10px;height:3px;"
            f"background:{color};margin-right:4px;vertical-align:middle'></span>"
            f"{html.escape(str(fila.route_label))}{tiempos}</div>"
        )
    contenido = "".join(elementos) or "Sin segmentos"
    # <details> permite plegar la leyenda con un clic sobre el título.
    leyenda = folium.Element(
        "<details open style='position:fixed;bottom:16px;left:16px;z-index:9999;"
        "background:white;border:1px solid #777;border-radius:4px;padding:4px 7px;"
        "font-size:10px;line-height:1.25;max-width:230px;max-height:150px;"
        "overflow-y:auto;box-shadow:0 1px 4px #999'>"
        f"<summary style='font-weight:bold;cursor:pointer'>{html.escape(titulo)}</summary>"
        "<div style='color:#555;margin:2px 0'>"
        "Min a Corregidora: P50 habitual · P80 confiable</div>"
        f"{contenido}</details>"
    )
    mapa.get_root().html.add_child(leyenda)


ESTILO_CONTROL_CAPAS_COMPACTO = """<style>
.leaflet-control-layers { font-size: 10px; line-height: 1.2; }
.leaflet-control-layers-expanded { padding: 3px 7px 3px 5px; }
.leaflet-control-layers label { margin: 0; }
.leaflet-control-layers-separator { margin: 2px -7px 2px -5px; }
.leaflet-control-layers-selector { transform: scale(0.8); margin: 0 2px 0 0; top: 2px; }
</style>"""


def agregar_control_capas(mapa):
    """Agrega el control de capas desplegado, en tamaño compacto."""
    folium.LayerControl(collapsed=False).add_to(mapa)
    mapa.get_root().header.add_child(folium.Element(ESTILO_CONTROL_CAPAS_COMPACTO))


## 4. Lectura y validación de paradas

Los identificadores se cargan como texto para conservar valores como `005`. Este notebook solo necesita `stops.txt` para dibujar el mapa; el resto del GTFS ya fue procesado por `01_PromedioRutas.ipynb`.


In [3]:
stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype={"stop_id": "string"})
routes = pd.read_csv(GTFS_DIR / "routes.txt", dtype={"route_id": "string"})
columnas_requeridas = {"stop_id", "stop_name", "stop_lat", "stop_lon"}
faltantes = columnas_requeridas - set(stops.columns)
if faltantes:
    raise ValueError(f"stops.txt no contiene: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    raise ValueError("stops.txt contiene stop_id duplicados.")
for columna in ["stop_lat", "stop_lon"]:
    stops[columna] = pd.to_numeric(stops[columna], errors="raise")
route_name_map = routes.set_index("route_id")["route_short_name"].to_dict()

print(f"{len(stops):,} paradas cargadas para construir el mapa.")


2,694 paradas cargadas para construir el mapa.


## 5. Carga del cálculo base

`01_PromedioRutas.ipynb` ya convirtió los horarios GTFS, construyó los segmentos y ejecutó Dijkstra con transbordos. Aquí se carga `data/processed/rutas_base_gtfs.csv`, se restauran las listas que forman cada camino y se valida que haya sido creado con la misma penalización configurada.


In [4]:
RUTAS_BASE_PATH = PROCESSED_DIR / "rutas_base_gtfs.csv"
if not RUTAS_BASE_PATH.exists():
    raise FileNotFoundError(
        "Falta data/processed/rutas_base_gtfs.csv. Ejecute 01_PromedioRutas.ipynb primero."
    )

resultados_base = cargar_rutas_base(RUTAS_BASE_PATH)
columnas_base_requeridas = {
    "stop_id",
    "tiempo_red_min",
    "num_transbordos",
    "tipo_conexion",
    "itinerario_route_ids",
    "itinerario_rutas",
    "paradas_transbordo",
    "camino_stop_ids",
    "rutas_por_segmento",
    "penalizacion_transbordo_config_min",
}
faltantes = columnas_base_requeridas - set(resultados_base.columns)
if faltantes:
    raise ValueError(f"El cálculo base no contiene: {sorted(faltantes)}")

penalizaciones_base = pd.to_numeric(
    resultados_base["penalizacion_transbordo_config_min"], errors="raise"
).dropna().unique()
if len(penalizaciones_base) != 1 or not np.isclose(
    penalizaciones_base[0], PENALIZACION_TRANSBORDO_MIN
):
    raise ValueError(
        "La penalización del cálculo base no coincide con .env. "
        "Vuelva a ejecutar 01_PromedioRutas.ipynb."
    )

TRAZOS_PATH = PROCESSED_DIR / "segmentos_red_vial.csv"
if not TRAZOS_PATH.exists():
    raise FileNotFoundError(
        "Falta data/processed/segmentos_red_vial.csv. Ejecute 01_PromedioRutas.ipynb primero."
    )
trazos_red_vial = cargar_trazos(TRAZOS_PATH)

print(f"Trazos sobre la red vial cargados: {len(trazos_red_vial):,} pares de paradas.")
print(
    f"Cálculo base cargado: {len(resultados_base):,} paradas; "
    "Dijkstra no se vuelve a ejecutar en este notebook."
)


Trazos sobre la red vial cargados: 3,204 pares de paradas.
Cálculo base cargado: 2,259 paradas; Dijkstra no se vuelve a ejecutar en este notebook.


## 6. Destino: estación Corregidora

Las coordenadas del proyecto se comparan con todas las paradas mediante distancia Haversine. El nodo destino del grafo es la parada QroBus más cercana. El marcador del mapa conserva las coordenadas exactas de la estación.


In [5]:
distancias_estacion = haversine_m(
    stops["stop_lat"].to_numpy(),
    stops["stop_lon"].to_numpy(),
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
)
target_idx = int(np.argmin(distancias_estacion))
target_row = stops.iloc[target_idx]
corregidora_stop_id = target_row["stop_id"]
distancia_estacion_stop_m = float(distancias_estacion[target_idx])

print(
    f"Parada destino: {corregidora_stop_id} — {target_row['stop_name']} "
    f"({distancia_estacion_stop_m:.0f} m de las coordenadas de la estación)."
)


Parada destino: 3025 — Estío/Calle Dr. Manuel Domínguez (211 m de las coordenadas de la estación).


## 7. Reutilización de los caminos mínimos

Los caminos mínimos con transbordos vienen completos desde `rutas_base_gtfs.csv`. Se verifica que el destino guardado sea la parada GTFS más cercana a Corregidora y después se continúa directamente con las correcciones de Google y la visualización.


In [6]:
resultados = resultados_base.copy()
destinos_base = resultados[resultados["tipo_conexion"].eq("Destino")]
if len(destinos_base) != 1:
    raise ValueError("El cálculo base debe contener exactamente un destino.")
if destinos_base.iloc[0]["stop_id"] != corregidora_stop_id:
    raise ValueError(
        "El destino del cálculo base no coincide con Corregidora. "
        "Vuelva a ejecutar 01_PromedioRutas.ipynb."
    )

resultados["distancia_red_km"] = resultados["camino_stop_ids"].apply(
    lambda camino: distancia_camino_km(camino, trazos_red_vial)
)

print(f"Paradas alcanzables cargadas: {len(resultados):,}")


Paradas alcanzables cargadas: 2,259


In [7]:
display(
    resultados[
        [
            "stop_id",
            "stop_name",
            "tiempo_red_min",
            "distancia_red_km",
            "num_transbordos",
            "itinerario_rutas",
        ]
    ].head(10)
)


,stop_id,stop_name,tiempo_red_min,distancia_red_km,num_transbordos,itinerario_rutas
0,3025,Estío/Calle Dr. Manuel Domínguez,0.000000,0.0000,0,NaN
1,2341,Av. Felipe Ángeles/Av. San Roque,1.533333,0.6000,0,C54
2,3295,Felipe Ángeles/Fraternidad,2.216667,0.8140,0,C54
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,2.733333,0.9754,0,C54
4,3294,Av. Felipe Ángeles/Calle del Porvenir,3.583333,1.2403,0,C54
5,2339,Av. Felipe Ángeles/Felipe Ángeles 225,4.200000,1.4324,0,C54
6,2338,Av. Felipe Ángeles/Estadística,4.700000,1.5872,0,C54
7,4256,Epigmenio González/Departamental Parques,5.366667,1.8336,0,C54
8,3363,Prol. Tecnológico/Calle San Joaquín,8.033333,2.8962,0,C54
9,3362,Prol. Tecnológico/Ford Citelis Querétaro,9.200000,3.2616,0,C54


## 8. Corrección opcional con observaciones de Google

La corrección se aplica únicamente cuando la fila está marcada como disponible. Se suma `atraso_promedio_ruta_min` al tiempo de red calculado aquí; no se reemplaza el camino ni se modifica el número de transbordos.

Las filas sin observación continúan como `GTFS programado`. Esta separación evita presentar datos no observados como si provinieran de Google.


In [8]:
ESTADISTICAS_STOP_PATH = PROCESSED_DIR / "estadisticas_historicas_google_por_parada.csv"
if not ESTADISTICAS_STOP_PATH.exists():
    raise FileNotFoundError(
        "Falta estadisticas_historicas_google_por_parada.csv. "
        "Ejecute 01_PromedioRutas.ipynb primero."
    )

estadisticas_stop = pd.read_csv(
    ESTADISTICAS_STOP_PATH, dtype={"stop_id": "string"}
)
requeridas_historico = {
    "stop_id", "p50_historico_min", "p80_historico_min",
    "observaciones", "proporcion_dentro_umbral",
    "ultima_eta_min", "ultima_eta_utc",
}
faltantes_historico = requeridas_historico - set(estadisticas_stop.columns)
if faltantes_historico:
    raise ValueError(
        f"Faltan columnas históricas: {sorted(faltantes_historico)}"
    )
for columna in [
    "p50_historico_min", "p80_historico_min", "observaciones",
    "proporcion_dentro_umbral", "ultima_eta_min",
    "observaciones_laboral", "observaciones_fin_de_semana",
    "p50_laboral_min", "p80_laboral_min",
    "p50_fin_de_semana_min", "p80_fin_de_semana_min",
]:
    estadisticas_stop[columna] = pd.to_numeric(
        estadisticas_stop[columna], errors="coerce"
    )
estadisticas_stop["ultima_eta_utc"] = pd.to_datetime(
    estadisticas_stop["ultima_eta_utc"], errors="coerce", utc=True
)
resultados["stop_id"] = resultados["stop_id"].astype(str)
resultados = resultados.merge(
    estadisticas_stop, on="stop_id", how="left", validate="one_to_one"
)
resultados["observaciones"] = resultados["observaciones"].fillna(0).astype(int)
for columna in ["observaciones_laboral", "observaciones_fin_de_semana"]:
    resultados[columna] = resultados[columna].fillna(0).astype(int)
muestra_suficiente = resultados["observaciones"].ge(
    MIN_OBSERVACIONES_HISTORICAS
)
resultados["clasificacion_acceso"] = np.select(
    [
        ~muestra_suficiente,
        resultados["p80_historico_min"].le(UMBRAL_ANALISIS_MIN),
        resultados["p50_historico_min"].le(UMBRAL_ANALISIS_MIN),
    ],
    [
        "Datos insuficientes",
        "Acceso confiable",
        "Acceso habitual pero variable",
    ],
    default="Fuera de 30 minutos",
)
resultados["edad_ultima_eta_horas"] = (
    pd.Timestamp.now(tz="UTC") - resultados["ultima_eta_utc"]
).dt.total_seconds() / 3600
resultados["ultima_eta_reciente"] = (
    resultados["ultima_eta_min"].notna()
    & resultados["edad_ultima_eta_horas"].between(
        -0.25, MAX_EDAD_ETA_HORAS, inclusive="both"
    )
)
# Alias para las celdas heredadas: el estimado conservador es P80, no una suma.
resultados["tiempo_estimado_min"] = resultados["p80_historico_min"]
resultados["fuente_tiempo"] = "P80 histórico de ETA Google por parada"
cobertura_historica = muestra_suficiente.loc[
    resultados["stop_id"].ne(corregidora_stop_id)
].mean()
print(
    f"Histórico robusto encontrado: {ESTADISTICAS_STOP_PATH.exists()}. "
    f"Cobertura con muestra suficiente: {cobertura_historica:.1%}."
)


Histórico robusto encontrado: True. Cobertura con muestra suficiente: 46.2%.


## 9. Resultados dentro de 30 minutos

La parada destino no se cuenta como origen. La clasificación principal usa el histórico general de ETA de Google por parada:

- **Acceso confiable:** muestra suficiente y P80 ≤ 30 minutos.
- **Acceso habitual pero variable:** muestra suficiente, P50 ≤ 30 y P80 > 30.
- **Datos insuficientes:** menos de `MIN_OBSERVACIONES_HISTORICAS`.
- **Situación actual:** última ETA reciente, mostrada en una capa separada.

### Por qué se usan percentiles

Un promedio esconde la variabilidad: quien planea con el tiempo promedio llega tarde aproximadamente la mitad de las veces. Por eso el análisis resume cada parada con dos percentiles de sus ETA observadas:

- **P50 (mediana):** el tiempo habitual; la mitad de las observaciones fueron iguales o menores.
- **P80:** el tiempo con margen; ocho de cada diez observaciones fueron iguales o menores.

El par P50/P80 es el que usa la Federal Highway Administration de Estados Unidos en su medida oficial de confiabilidad, *Level of Travel Time Reliability* (LOTTR), definida como el tiempo de viaje del percentil 80 dividido entre el tiempo "normal" del percentil 50 ([23 CFR § 490.511](https://www.law.cornell.edu/cfr/text/23/490.511); [preguntas frecuentes de la FHWA](https://www.fhwa.dot.gov/tpm/faq.cfm)). La misma agencia fundamenta el uso de percentiles en lugar de promedios para describir la confiabilidad en [*Travel Time Reliability: Making It There On Time, All The Time*](https://ops.fhwa.dot.gov/publications/tt_reliability/TTR_Report.htm), donde el tiempo de planeación se calcula con el percentil 95 y se admiten otros percentiles según el nivel de confiabilidad buscado.

> Esa norma se diseñó para tramos carreteros con muchas mediciones. Aquí se adopta por analogía para ETA de transporte público con muestras pequeñas (entre 1 y 9 observaciones por parada), de modo que el P80 es una estimación preliminar y no un indicador regulatorio.

En los mapas, cada tramo muestra el P50 y el P80 hacia Corregidora de la parada donde inicia, y la leyenda resume el rango de cada ruta.

Las estadísticas laboral/fin de semana permanecen en la tabla para contexto y segmentación futura, pero todavía no filtran la clasificación.


In [9]:
es_origen = resultados["stop_id"].ne(corregidora_stop_id)
paradas_30 = resultados[
    es_origen
    & resultados["clasificacion_acceso"].eq("Acceso confiable")
].copy()
paradas_variables_30 = resultados[
    es_origen
    & resultados["clasificacion_acceso"].eq(
        "Acceso habitual pero variable"
    )
].copy()
paradas_insuficientes_30 = resultados[
    es_origen
    & resultados["clasificacion_acceso"].eq("Datos insuficientes")
    & resultados["tiempo_red_min"].le(UMBRAL_ANALISIS_MIN)
].copy()
paradas_actuales_30 = resultados[
    es_origen
    & resultados["ultima_eta_reciente"]
    & resultados["ultima_eta_min"].le(UMBRAL_ANALISIS_MIN)
].copy()

paradas_directas_30 = paradas_30[
    paradas_30["num_transbordos"].eq(0)
].copy()
paradas_con_transbordo_30 = paradas_30[
    paradas_30["num_transbordos"].gt(0)
].copy()

tabla_itinerarios_30 = (
    paradas_30.groupby(
        [
            "itinerario_route_ids",
            "itinerario_rutas",
            "tipo_conexion",
            "num_transbordos",
        ],
        as_index=False,
    )
    .agg(
        paradas_origen=("stop_id", "nunique"),
        tiempo_minimo_min=("tiempo_estimado_min", "min"),
        tiempo_maximo_min=("tiempo_estimado_min", "max"),
        tiempo_promedio_min=("tiempo_estimado_min", "mean"),
    )
    .sort_values(
        ["num_transbordos", "tiempo_minimo_min", "itinerario_rutas"]
    )
    .reset_index(drop=True)
)

rutas_por_parada = paradas_30[[
    "stop_id", "itinerario_rutas", "p50_historico_min",
    "p80_historico_min", "observaciones",
]].copy()
rutas_por_parada["ruta"] = rutas_por_parada["itinerario_rutas"].str.split(" → " )
rutas_por_parada = (
    rutas_por_parada.explode("ruta")
    .dropna(subset=["ruta"])
    .drop_duplicates(["stop_id", "ruta"])
)
tabla_rutas_acceso = (
    rutas_por_parada.groupby("ruta", as_index=False)
    .agg(
        paradas_que_utilizan_ruta=("stop_id", "nunique"),
        p50_minimo_min=("p50_historico_min", "min"),
        p50_maximo_min=("p50_historico_min", "max"),
        p80_minimo_min=("p80_historico_min", "min"),
        p80_maximo_min=("p80_historico_min", "max"),
        observaciones_mediana=("observaciones", "median"),
    )
    .sort_values(["paradas_que_utilizan_ruta", "ruta"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Acceso confiable (P80 <= {UMBRAL_ANALISIS_MIN:g}): {len(paradas_30):,}")
print(f"Acceso habitual pero variable: {len(paradas_variables_30):,}")
print(f"Datos insuficientes entre candidatas GTFS: {len(paradas_insuficientes_30):,}")
print(f"Última ETA reciente <= {UMBRAL_ANALISIS_MIN:g}: {len(paradas_actuales_30):,}")
print(f"  Conexión directa: {len(paradas_directas_30):,}")
print(f"  Con transbordos: {len(paradas_con_transbordo_30):,}")
print(f"Itinerarios de rutas distintos: {len(tabla_itinerarios_30):,}")


Acceso confiable (P80 <= 30): 327
Acceso habitual pero variable: 36
Datos insuficientes entre candidatas GTFS: 0
Última ETA reciente <= 30: 0
  Conexión directa: 15
  Con transbordos: 312
Itinerarios de rutas distintos: 79


### 9.1 Rutas e itinerarios que llegan dentro del umbral

Cada fila representa una secuencia de rutas, no una geometría duplicada por cada parada. `C54` indica un viaje directo; `CXX → C54` indica un transbordo.


In [10]:
columnas_itinerarios = [
    "itinerario_rutas",
    "tipo_conexion",
    "num_transbordos",
    "paradas_origen",
    "tiempo_minimo_min",
    "tiempo_promedio_min",
    "tiempo_maximo_min",
]
display(
    tabla_itinerarios_30[columnas_itinerarios].head().style.format(
        {
            "tiempo_minimo_min": "{:.1f}",
            "tiempo_promedio_min": "{:.1f}",
            "tiempo_maximo_min": "{:.1f}",
        }
    )
)


,itinerario_rutas,tipo_conexion,num_transbordos,paradas_origen,tiempo_minimo_min,tiempo_promedio_min,tiempo_maximo_min
0,C54,Directa,0,15,9.8,15.6,21.9
1,C33 → C54,Con transbordo,1,22,10.5,19.7,28.6
2,L154 → C54,Con transbordo,1,10,15.8,22.3,29.6
3,T07 → C54,Con transbordo,1,3,18.1,24.0,27.7
4,C62 → C54,Con transbordo,1,5,18.5,22.6,29.1


### 9.2 Paradas dentro del umbral, incluyendo transbordos

La tabla conserva el itinerario y las estadísticas generales. Las columnas laboral/fin de semana aportan contexto sobre la composición de la muestra, pero no se usan todavía para filtrar el resultado.


In [11]:
tabla_paradas_30 = paradas_30[
    [
        "stop_id",
        "stop_name",
        "tipo_conexion",
        "itinerario_rutas",
        "paradas_transbordo",
        "num_transbordos",
        "num_paradas_camino",
        "tiempo_red_min",
        "p50_historico_min",
        "p80_historico_min",
        "observaciones",
        "proporcion_dentro_umbral",
        "observaciones_laboral",
        "observaciones_fin_de_semana",
        "p50_laboral_min",
        "p50_fin_de_semana_min",
        "ultima_eta_min",
        "ultimo_tipo_dia",
        "ultima_franja_horaria",
        "calidad_muestra",
        "clasificacion_acceso",
    ]
].sort_values(["p80_historico_min", "stop_name"])

PARADAS_ACCESO_PATH = PROCESSED_DIR / "paradas_acceso_corregidora.csv"
ITINERARIOS_ACCESO_PATH = PROCESSED_DIR / "itinerarios_acceso_corregidora.csv"
RUTAS_ACCESO_PATH = PROCESSED_DIR / "rutas_acceso_corregidora.csv"
tabla_paradas_30.to_csv(PARADAS_ACCESO_PATH, index=False)
tabla_itinerarios_30.to_csv(ITINERARIOS_ACCESO_PATH, index=False)
tabla_rutas_acceso.to_csv(RUTAS_ACCESO_PATH, index=False)
print(f"Paradas exportadas: {PARADAS_ACCESO_PATH.resolve()}")
print(f"Itinerarios exportados: {ITINERARIOS_ACCESO_PATH.resolve()}")
print(f"Rutas exportadas: {RUTAS_ACCESO_PATH.resolve()}")

display(
    tabla_paradas_30.head().style.format(
        {
            "tiempo_red_min": "{:.1f}",
            "p50_historico_min": "{:.1f}",
            "p80_historico_min": "{:.1f}",
            "proporcion_dentro_umbral": "{:.1%}",
            "p50_laboral_min": "{:.1f}",
            "p50_fin_de_semana_min": "{:.1f}",
            "ultima_eta_min": "{:.1f}",
        }
    )
)


Paradas exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/paradas_acceso_corregidora.csv
Itinerarios exportados: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/itinerarios_acceso_corregidora.csv
Rutas exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/rutas_acceso_corregidora.csv


,stop_id,stop_name,tipo_conexion,itinerario_rutas,paradas_transbordo,num_transbordos,num_paradas_camino,tiempo_red_min,p50_historico_min,p80_historico_min,observaciones,proporcion_dentro_umbral,observaciones_laboral,observaciones_fin_de_semana,p50_laboral_min,p50_fin_de_semana_min,ultima_eta_min,ultimo_tipo_dia,ultima_franja_horaria,calidad_muestra,clasificacion_acceso
811,5200,Calle Nicolás Bravo/Calle Primavera,Con transbordo,T07 → T03 → C62 → C54,Río Tuxpan → Av. Tláloc/Av. Pie de la Cuesta → Av. Cerro Sombrerete/Av. Playa Roqueta,3,28,44.7,5.0,5.0,10,100.0%,9,1,5.0,5.0,5.0,laboral,19-23,Alta,Acceso confiable
419,3263,San Agustín del Retablo/Salvador Galván,Con transbordo,C53 → C46 → C33 → C54,Ezequiel Montes/Mariano Escobedo → Avenida Ignacio Zaragoza/Comercial Mexicana → Epigmenio González/Departamental Parques,3,24,37.2,6.7,6.7,10,100.0%,9,1,6.7,6.7,6.7,laboral,19-23,Alta,Acceso confiable
373,5193,San Agustín del Retablo/Estio,Con transbordo,C53 → C46 → C33 → C54,Ezequiel Montes/Mariano Escobedo → Avenida Ignacio Zaragoza/Comercial Mexicana → Epigmenio González/Departamental Parques,3,23,36.2,6.9,6.9,10,100.0%,9,1,6.9,6.9,6.9,laboral,19-23,Alta,Acceso confiable
418,2680,Av. Universidad/Calle Ezequiel Montes,Con transbordo,C55 → C34 → C62 → C54,Av. Universidad/Prol. Tecnológico Norte → San Pablo/Loma de San Pablo → Prol. Tecnológico/Ford Citelis Querétaro,3,20,37.2,7.3,7.3,10,100.0%,9,1,7.3,7.3,7.3,laboral,19-23,Alta,Acceso confiable
861,2316,Calle Nicolás Bravo/Calle Segunda de Las Rosas,Con transbordo,T07 → T03 → C62 → C54,Río Tuxpan → Av. Tláloc/Av. Pie de la Cuesta → Av. Cerro Sombrerete/Av. Playa Roqueta,3,29,45.6,7.9,7.9,10,100.0%,9,1,7.9,7.9,7.9,laboral,19-23,Alta,Acceso confiable


## 10. Pruebas automáticas de calidad

Estas comprobaciones validan que los caminos terminen en Corregidora, que los tiempos sean no negativos, que los transbordos coincidan con los cambios de ruta y que ninguna parada del reporte exceda el umbral.


In [12]:
assert corregidora_stop_id in set(resultados["stop_id"])
assert resultados["stop_id"].is_unique
assert np.isfinite(resultados["tiempo_red_min"]).all()
assert resultados["tiempo_red_min"].ge(0).all()
assert paradas_30["observaciones"].ge(MIN_OBSERVACIONES_HISTORICAS).all()
assert paradas_30["p80_historico_min"].le(
    UMBRAL_ANALISIS_MIN + 1e-9
).all()
assert paradas_30["stop_id"].ne(corregidora_stop_id).all()

for fila in resultados.itertuples(index=False):
    assert fila.camino_stop_ids[-1] == corregidora_stop_id
    rutas_compactas = compactar_rutas(fila.rutas_por_segmento)
    assert fila.num_transbordos == max(len(rutas_compactas) - 1, 0)
    assert len(fila.camino_stop_ids) == len(fila.rutas_por_segmento) + 1

print("QA OK: todas las validaciones automáticas fueron superadas.")


QA OK: todas las validaciones automáticas fueron superadas.


## 11. Mapa principal de accesibilidad

El mapa consolida las visualizaciones anteriores:

- **Color de relleno:** ruta inicial utilizada desde cada parada.
- **Borde verde:** acceso confiable según P80; **naranja:** acceso habitual pero variable.
- **Borde gris:** muestra histórica insuficiente; se oculta inicialmente.
- **Anillo azul:** última ETA reciente dentro del umbral; es contexto actual y se oculta inicialmente.
- **Rojo:** estación Corregidora y su parada GTFS más cercana.
- **Líneas:** todos los segmentos realmente utilizados, deduplicados, coloreados por ruta y trazados sobre las calles de OpenStreetMap. Al pasar el cursor se muestra la distancia del tramo por calles.

El control de capas permite ocultar o mostrar cada grupo sin crear mapas inconsistentes.


In [13]:
coords_map = stops.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")
paradas_visualizacion = pd.concat(
    [paradas_30, paradas_variables_30, paradas_insuficientes_30, paradas_actuales_30],
    ignore_index=True,
).drop_duplicates("stop_id")
segmentos_mapa_30 = extraer_segmentos_conectados(
    paradas_visualizacion, coords_map, route_name_map, trazos_red_vial,
    tiempos_historicos_por_parada(resultados),
)
paradas_mapa_30, aisladas_confiables = separar_origenes_conectados(
    paradas_30, segmentos_mapa_30
)
paradas_mapa_variables, aisladas_variables = separar_origenes_conectados(
    paradas_variables_30, segmentos_mapa_30
)
paradas_mapa_insuficientes, aisladas_insuficientes = separar_origenes_conectados(
    paradas_insuficientes_30, segmentos_mapa_30
)
paradas_mapa_actuales, aisladas_actuales = separar_origenes_conectados(
    paradas_actuales_30, segmentos_mapa_30
)

mapa_30 = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON], zoom_start=13, tiles="OpenStreetMap"
)
capa_trayectos = folium.FeatureGroup(
    name=f"Segmentos conectados ({len(segmentos_mapa_30)})", show=True
)
capa_confiables = folium.FeatureGroup(
    name=f"Acceso confiable P80 ({len(paradas_mapa_30)})", show=True
)
capa_variables = folium.FeatureGroup(
    name=f"Acceso habitual variable ({len(paradas_mapa_variables)})", show=True
)
capa_insuficientes = folium.FeatureGroup(
    name=f"Datos insuficientes ({len(paradas_mapa_insuficientes)})", show=False
)
capa_actual = folium.FeatureGroup(
    name=f"Última ETA <= 30 ({len(paradas_mapa_actuales)})", show=False
)
agregar_segmentos_rutas(capa_trayectos, segmentos_mapa_30)

folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    tooltip="Estación Corregidora",
    popup=(
        f"<b>Estación Corregidora</b><br>"
        f"Parada GTFS cercana: {target_row['stop_name']}<br>"
        f"Distancia: {distancia_estacion_stop_m:.0f} m"
    ),
    icon=folium.Icon(color="red", icon="flag"),
).add_to(mapa_30)
folium.CircleMarker(
    [target_row["stop_lat"], target_row["stop_lon"]],
    radius=5, color="darkred", fill=True, fill_opacity=0.9,
    tooltip=f"Parada destino GTFS: {target_row['stop_name']}",
).add_to(mapa_30)

def agregar_paradas_historicas(df, capa, color_borde):
    for fila in df.itertuples(index=False):
        route_id_inicio = ruta_inicial(fila.rutas_por_segmento)
        folium.CircleMarker(
            [fila.stop_lat, fila.stop_lon], radius=5,
            color=color_borde, weight=3, fill=True,
            fill_color=color_por_ruta(route_id_inicio), fill_opacity=0.9,
            tooltip=(
                f"{fila.stop_name} | {fila.clasificacion_acceso} | "
                f"P50 {fila.p50_historico_min:.1f} min | "
                f"P80 {fila.p80_historico_min:.1f} min | "
                f"n={fila.observaciones} | "
                f"<=30: {fila.proporcion_dentro_umbral:.0%} | "
                f"{fila.distancia_red_km:.1f} km por calles | "
                f"{fila.itinerario_rutas}"
            ),
        ).add_to(capa)

agregar_paradas_historicas(paradas_mapa_30, capa_confiables, "#15803d")
agregar_paradas_historicas(paradas_mapa_variables, capa_variables, "#f97316")
agregar_paradas_historicas(paradas_mapa_insuficientes, capa_insuficientes, "#6b7280")
for fila in paradas_mapa_actuales.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon], radius=8, color="#2563eb",
        weight=3, fill=False,
        tooltip=(
            f"{fila.stop_name} | última ETA {fila.ultima_eta_min:.1f} min | "
            f"histórico: {fila.clasificacion_acceso}"
        ),
    ).add_to(capa_actual)

capa_trayectos.add_to(mapa_30)
capa_confiables.add_to(mapa_30)
capa_variables.add_to(mapa_30)
capa_insuficientes.add_to(mapa_30)
capa_actual.add_to(mapa_30)
agregar_leyenda_rutas(mapa_30, segmentos_mapa_30, "Rutas del mapa principal")
agregar_control_capas(mapa_30)
mapa_30.fit_bounds([
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    *paradas_visualizacion[["stop_lat", "stop_lon"]].to_numpy().tolist(),
])

assert set(paradas_mapa_30["stop_id"]).issubset(set(segmentos_mapa_30["stop_id_a"]))
print(f"Segmentos únicos dibujados: {len(segmentos_mapa_30):,}.")
paradas_aisladas_30 = pd.concat(
    [aisladas_confiables, aisladas_variables, aisladas_insuficientes, aisladas_actuales],
    ignore_index=True,
).drop_duplicates("stop_id")
print(f"Paradas omitidas por no tener segmento dibujable: {len(paradas_aisladas_30):,}.")

MAPA_PATH = MAPS_DIR / "mapa_rutas_qrobus.html"
mapa_30.save(MAPA_PATH)
print(f"Copia HTML: {MAPA_PATH.resolve()}")
# La última expresión también integra el mapa directamente en el notebook.
mapa_30


Segmentos únicos dibujados: 642.
Paradas omitidas por no tener segmento dibujable: 0.


Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_rutas_qrobus.html


## 12. Todas las paradas a menos de 30 minutos sin cruzar las vías

Este escenario carga el segundo Dijkstra generado por `01_PromedioRutas.ipynb` después de retirar del grafo los segmentos que cruzan la `Línea Juárez`. La parada de Mariano Escobedo solo define el lado sur de referencia; el resultado incluye **todas** las paradas que todavía pueden llegar a Corregidora dentro del umbral mediante autobús y un último tramo a pie de diez minutos o menos, medido sobre la red peatonal de OpenStreetMap del lado sur.

La clasificación vuelve a unir las estadísticas históricas por `stop_id`, no por ruta: P80 define acceso confiable, P50 identifica acceso variable y la última ETA queda como una capa opcional. La polilínea ferroviaria procede de OpenStreetMap; datos cartográficos © [colaboradores de OpenStreetMap](https://www.openstreetmap.org/copyright), ODbL.


In [14]:
RUTAS_SUR_VIAS_PATH = PROCESSED_DIR / "rutas_base_sur_vias.csv"
DESTINO_PEATONAL_SUR_ID = "ESTACION_CORREGIDORA_SUR"
if not RUTAS_SUR_VIAS_PATH.exists():
    raise FileNotFoundError(
        "Falta data/processed/rutas_base_sur_vias.csv. Ejecute 01_PromedioRutas.ipynb primero."
    )

resultados_sur_vias = cargar_rutas_base(RUTAS_SUR_VIAS_PATH)
destino_sur = resultados_sur_vias[
    resultados_sur_vias["tipo_conexion"].eq("Destino")
]
if (
    len(destino_sur) != 1
    or destino_sur.iloc[0]["stop_id"] != DESTINO_PEATONAL_SUR_ID
):
    raise ValueError(
        "El escenario sin cruces no corresponde al destino actual. "
        "Vuelva a ejecutar 01_PromedioRutas.ipynb."
    )
penalizacion_sur = pd.to_numeric(
    resultados_sur_vias["penalizacion_transbordo_config_min"],
    errors="raise",
).dropna().unique()
if len(penalizacion_sur) != 1 or not np.isclose(
    penalizacion_sur[0], PENALIZACION_TRANSBORDO_MIN
):
    raise ValueError(
        "La penalización del escenario sin cruces no coincide con .env."
    )

resultados_sur_vias["stop_id"] = resultados_sur_vias["stop_id"].astype(str)
resultados_sur_vias["distancia_red_km"] = resultados_sur_vias[
    "camino_stop_ids"
].apply(lambda camino: distancia_camino_km(camino, trazos_red_vial))
resultados_sur_vias = resultados_sur_vias.merge(
    estadisticas_stop, on="stop_id", how="left", validate="one_to_one"
)
resultados_sur_vias["observaciones"] = resultados_sur_vias[
    "observaciones"
].fillna(0).astype(int)
muestra_suficiente_sur = resultados_sur_vias["observaciones"].ge(
    MIN_OBSERVACIONES_HISTORICAS
)
resultados_sur_vias["clasificacion_acceso"] = np.select(
    [
        ~muestra_suficiente_sur,
        resultados_sur_vias["p80_historico_min"].le(UMBRAL_ANALISIS_MIN),
        resultados_sur_vias["p50_historico_min"].le(UMBRAL_ANALISIS_MIN),
    ],
    ["Datos insuficientes", "Acceso confiable", "Acceso habitual pero variable"],
    default="Fuera de 30 minutos",
)
resultados_sur_vias["edad_ultima_eta_horas"] = (
    pd.Timestamp.now(tz="UTC") - resultados_sur_vias["ultima_eta_utc"]
).dt.total_seconds() / 3600
resultados_sur_vias["ultima_eta_reciente"] = (
    resultados_sur_vias["ultima_eta_min"].notna()
    & resultados_sur_vias["edad_ultima_eta_horas"].between(
        -0.25, MAX_EDAD_ETA_HORAS, inclusive="both"
    )
)
resultados_sur_vias["tiempo_estimado_min"] = resultados_sur_vias[
    "p80_historico_min"
]

candidatas_modelo_30 = resultados_sur_vias[
    resultados_sur_vias["stop_id"].ne(DESTINO_PEATONAL_SUR_ID)
    & resultados_sur_vias["tiempo_red_min"].le(UMBRAL_ANALISIS_MIN)
].copy()
paradas_sur_vias_30 = candidatas_modelo_30[
    candidatas_modelo_30["clasificacion_acceso"].eq("Acceso confiable")
].copy()
paradas_sur_variables = candidatas_modelo_30[
    candidatas_modelo_30["clasificacion_acceso"].eq(
        "Acceso habitual pero variable"
    )
].copy()
paradas_sur_insuficientes = candidatas_modelo_30[
    candidatas_modelo_30["clasificacion_acceso"].eq("Datos insuficientes")
].copy()
paradas_sur_actuales = candidatas_modelo_30[
    candidatas_modelo_30["ultima_eta_reciente"]
    & candidatas_modelo_30["ultima_eta_min"].le(UMBRAL_ANALISIS_MIN)
].copy()

print(
    f"Acceso confiable al sur (P80 <= {UMBRAL_ANALISIS_MIN:g}): "
    f"{len(paradas_sur_vias_30):,}."
)
print(f"Acceso habitual pero variable: {len(paradas_sur_variables):,}.")
print(f"Datos insuficientes: {len(paradas_sur_insuficientes):,}.")
print(f"Última ETA reciente <= {UMBRAL_ANALISIS_MIN:g}: {len(paradas_sur_actuales):,}.")
print(
    "La parada 2312 solo se utilizó para elegir el lado de la vía; "
    "no limita los orígenes del resultado."
)
columnas_sur_vias = [
    "stop_id",
    "stop_name",
    "tipo_conexion",
    "itinerario_rutas",
    "num_transbordos",
    "tiempo_red_min",
    "distancia_red_km",
    "p50_historico_min",
    "p80_historico_min",
    "observaciones",
    "proporcion_dentro_umbral",
    "observaciones_laboral",
    "observaciones_fin_de_semana",
    "p50_laboral_min",
    "p50_fin_de_semana_min",
    "ultima_eta_min",
    "ultimo_tipo_dia",
    "ultima_franja_horaria",
    "clasificacion_acceso",
]
display(
    paradas_sur_vias_30[columnas_sur_vias].head().style.format(
        {
            "tiempo_red_min": "{:.1f}",
            "distancia_red_km": "{:.2f}",
            "p50_historico_min": "{:.1f}",
            "p80_historico_min": "{:.1f}",
            "proporcion_dentro_umbral": "{:.1%}",
            "p50_laboral_min": "{:.1f}",
            "p50_fin_de_semana_min": "{:.1f}",
            "ultima_eta_min": "{:.1f}",
        }
    )
)

coords_sur = resultados_sur_vias.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")
paradas_visualizacion_sur = pd.concat(
    [paradas_sur_vias_30, paradas_sur_variables, paradas_sur_insuficientes, paradas_sur_actuales],
    ignore_index=True,
).drop_duplicates("stop_id")
segmentos_sur_vias = extraer_segmentos_conectados(
    paradas_visualizacion_sur, coords_sur, route_name_map, trazos_red_vial,
    tiempos_historicos_por_parada(resultados_sur_vias),
)
paradas_mapa_sur, aisladas_sur_confiables = separar_origenes_conectados(
    paradas_sur_vias_30, segmentos_sur_vias
)
paradas_mapa_sur_variables, aisladas_sur_variables = separar_origenes_conectados(
    paradas_sur_variables, segmentos_sur_vias
)
paradas_mapa_sur_insuficientes, aisladas_sur_insuficientes = separar_origenes_conectados(
    paradas_sur_insuficientes, segmentos_sur_vias
)
paradas_mapa_sur_actuales, aisladas_sur_actuales = separar_origenes_conectados(
    paradas_sur_actuales, segmentos_sur_vias
)

mapa_sur_vias = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON], zoom_start=13, tiles="OpenStreetMap"
)
capa_rutas_sur = folium.FeatureGroup(
    name=f"Segmentos conectados ({len(segmentos_sur_vias)})", show=True
)
capa_paradas_sur = folium.FeatureGroup(
    name=f"Acceso confiable P80 ({len(paradas_mapa_sur)})", show=True
)
capa_sur_variables = folium.FeatureGroup(
    name=f"Acceso habitual variable ({len(paradas_mapa_sur_variables)})", show=True
)
capa_sur_insuficientes = folium.FeatureGroup(
    name=f"Datos insuficientes ({len(paradas_mapa_sur_insuficientes)})", show=False
)
capa_sur_actual = folium.FeatureGroup(
    name=f"Última ETA <= 30 ({len(paradas_mapa_sur_actuales)})", show=False
)
agregar_segmentos_rutas(capa_rutas_sur, segmentos_sur_vias)
folium.PolyLine(
    VIA_FERREA_CORREGIDORA_LATLON,
    color="black",
    weight=5,
    opacity=0.85,
    tooltip="Línea ferroviaria usada como barrera",
).add_to(mapa_sur_vias)
folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    tooltip="Estación Corregidora",
    icon=folium.Icon(color="red", icon="train", prefix="fa"),
).add_to(mapa_sur_vias)

agregar_paradas_historicas(paradas_mapa_sur, capa_paradas_sur, "#15803d")
agregar_paradas_historicas(
    paradas_mapa_sur_variables, capa_sur_variables, "#f97316"
)
agregar_paradas_historicas(
    paradas_mapa_sur_insuficientes, capa_sur_insuficientes, "#6b7280"
)
for fila in paradas_mapa_sur_actuales.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon], radius=8, color="#2563eb",
        weight=3, fill=False,
        tooltip=(
            f"{fila.stop_name} | última ETA {fila.ultima_eta_min:.1f} min | "
            f"histórico: {fila.clasificacion_acceso}"
        ),
    ).add_to(capa_sur_actual)

capa_rutas_sur.add_to(mapa_sur_vias)
capa_paradas_sur.add_to(mapa_sur_vias)
capa_sur_variables.add_to(mapa_sur_vias)
capa_sur_insuficientes.add_to(mapa_sur_vias)
capa_sur_actual.add_to(mapa_sur_vias)
agregar_leyenda_rutas(mapa_sur_vias, segmentos_sur_vias, "Rutas sin cruzar vías")
agregar_control_capas(mapa_sur_vias)
mapa_sur_vias.fit_bounds(
    [
        [CORREGIDORA_LAT, CORREGIDORA_LON],
        *paradas_visualizacion_sur[["stop_lat", "stop_lon"]].to_numpy().tolist(),
    ]
)
assert set(paradas_mapa_sur["stop_id"]).issubset(set(segmentos_sur_vias["stop_id_a"]))
print(f"Segmentos únicos sin cruce dibujados: {len(segmentos_sur_vias):,}.")
paradas_aisladas_sur = pd.concat(
    [aisladas_sur_confiables, aisladas_sur_variables, aisladas_sur_insuficientes, aisladas_sur_actuales],
    ignore_index=True,
).drop_duplicates("stop_id")
print(f"Paradas omitidas por no tener segmento dibujable: {len(paradas_aisladas_sur):,}.")
MAPA_SUR_VIAS_PATH = MAPS_DIR / "mapa_paradas_sur_vias_30_min.html"
mapa_sur_vias.save(MAPA_SUR_VIAS_PATH)
print(f"Copia HTML: {MAPA_SUR_VIAS_PATH.resolve()}")
# La última expresión también integra el mapa directamente en el notebook.
mapa_sur_vias


Acceso confiable al sur (P80 <= 30): 126.
Acceso habitual pero variable: 10.
Datos insuficientes: 1.
Última ETA reciente <= 30: 0.
La parada 2312 solo se utilizó para elegir el lado de la vía; no limita los orígenes del resultado.


,stop_id,stop_name,tipo_conexion,itinerario_rutas,num_transbordos,tiempo_red_min,distancia_red_km,p50_historico_min,p80_historico_min,observaciones,proporcion_dentro_umbral,observaciones_laboral,observaciones_fin_de_semana,p50_laboral_min,p50_fin_de_semana_min,ultima_eta_min,ultimo_tipo_dia,ultima_franja_horaria,clasificacion_acceso
2,5200,Calle Nicolás Bravo/Calle Primavera,Solo caminata,nan,0,5.6,0.45,5.0,5.0,10,100.0%,9.000000,1.000000,5.0,5.0,5.0,laboral,19-23,Acceso confiable
3,2316,Calle Nicolás Bravo/Calle Segunda de Las Rosas,Directa,C55,0,6.3,0.66,7.9,7.9,10,100.0%,9.000000,1.000000,7.9,7.9,7.9,laboral,19-23,Acceso confiable
4,5194,Av. Universidad/Calle Rafael Osuna,Directa,C55,0,6.8,0.83,8.5,8.5,10,100.0%,9.000000,1.000000,8.5,8.5,8.5,laboral,19-23,Acceso confiable
5,2394,Av. Universidad/Calle Nicolás Bravo,Directa,C54,0,6.8,0.73,8.9,8.9,10,100.0%,9.000000,1.000000,8.9,8.9,8.9,laboral,19-23,Acceso confiable
6,2315,Av. Universidad/San Andrés,Directa,C55,0,7.2,0.98,9.0,9.0,10,100.0%,9.000000,1.000000,9.0,9.0,9.0,laboral,19-23,Acceso confiable


Segmentos únicos sin cruce dibujados: 242.
Paradas omitidas por no tener segmento dibujable: 0.
Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_paradas_sur_vias_30_min.html


## 13. Demanda potencial con la malla geoestadística

La demanda se estima con la **malla geoestadística del INEGI, nivel 9** (`data/InegiMalla/`), proporcionada por el Instituto de Movilidad. Es una cuadrícula uniforme con los indicadores del Censo 2020: cada celda mide unos 468 m por lado (0.219 km², unas 22 hectáreas). Se eligió en lugar de las manzanas del servicio del INEGI porque ambas fuentes dan resultados muy parecidos y la malla no depende de descargas externas.

- `Datos totales.csv`: 38,937 celdas de Querétaro y zonas vecinas.
- `Datos filtrados.csv`: las seis celdas que rodean la estación Corregidora; se usan para validar la geometría.

**Método.** Alrededor de cada parada conectada se traza un buffer de 800 metros de diámetro (`RADIO_DEMANDA_M=400`, el radio, en `.env`), unos cinco minutos caminando. Como los buffers cortan las celdas en fracciones, el valor principal es el **prorrateo por área**: de cada celda se cuenta la proporción de su superficie dentro de la unión de buffers, suponiendo población uniforme en la celda. La suma de celdas completas se muestra solo como techo. Es una estimación, no la localización exacta de viviendas.

**Cuidados con estos datos.**

- Los valores negativos (`-6`, `-7`, `-9`) son códigos del INEGI para datos reservados o no disponibles; se tratan como vacíos y no suman.
- La malla publica población de **60 años o más** (`P_60YMAS`), no de 65 o más.
- No trae mujeres y hombres por separado. Se estiman con la relación hombres-mujeres (`REL_H_M`, hombres por cada 100 mujeres) de cada celda.
- Los CSV no incluyen geometría. `src/scripts/malla_inegi.py` reconstruye cada celda desde su código rHEALPix con una corrección de latitud calibrada contra las manzanas del censo (correlación de 0.99 entre la población de la celda y la de las manzanas que contiene). Si se consigue el shapefile oficial de la malla, conviene usarlo en su lugar.

In [15]:
import json

import geopandas as gpd
import shapely

CRS_METRICO_QUERETARO = "EPSG:32614"
# Población total del estado de Querétaro en el Censo de Población y Vivienda 2020.
POBLACION_ESTATAL_CENSO_2020 = 2_368_467
FUENTE_MALLA = (
    "Malla geoestadística del INEGI, nivel 9, con indicadores del Censo 2020; "
    "proporcionada por el Instituto de Movilidad"
)
# Indicadores que se prorratean por área. La malla publica 60 años o más (no 65)
# y no separa mujeres y hombres: se estiman con REL_H_M (hombres por cada 100 mujeres).
INDICADORES_MALLA = {
    "POBTOT": "Personas",
    "mujeres_estimadas": "Mujeres (estimadas)",
    "hombres_estimados": "Hombres (estimados)",
    "POB0_14": "Niños (0-14 años)",
    "P_60YMAS": "Mayores (60 años o más)",
    "PCON_DISC": "Personas con discapacidad",
    "PEA": "Personas activas económicamente",
    "TVIVHAB": "Viviendas habitadas",
}


def cobertura_buffers(paradas, radio_m):
    """Unión de los buffers de las paradas, en el CRS métrico."""
    return (
        gpd.GeoSeries(
            gpd.points_from_xy(paradas["stop_lon"], paradas["stop_lat"]), crs="EPSG:4326"
        )
        .to_crs(CRS_METRICO_QUERETARO)
        .buffer(radio_m)
        .union_all()
    )


def estimar_demanda(malla_m, paradas, radio_m):
    """Cruza la unión de buffers con la malla y prorratea cada indicador por área.

    Devuelve las celdas tocadas en EPSG:4326 y un resumen. El valor principal es
    el prorrateo: de cada celda se cuenta la fracción de su superficie dentro
    de los buffers, suponiendo población uniforme en la celda. La suma de
    celdas completas se reporta solo como techo.
    """
    if paradas.empty:
        vacias = malla_m.iloc[0:0].assign(
            fraccion_area_cubierta=np.nan,
            **{f"{columna}_prorrateada": np.nan for columna in INDICADORES_MALLA},
        )
        return vacias.to_crs("EPSG:4326"), None
    cobertura_m = cobertura_buffers(paradas, radio_m)
    tocadas = estimar_demanda_malla(malla_m, cobertura_m, list(INDICADORES_MALLA))
    poblacion_prorrateada = float(tocadas["POBTOT_prorrateada"].sum())
    resumen = {
        "fuente": FUENTE_MALLA,
        "metodo": "prorrateo por área de las celdas de la malla tocadas por los buffers",
        "radio_buffer_m": radio_m,
        "numero_paradas": int(len(paradas)),
        "numero_celdas_malla": int(len(malla_m)),
        "numero_celdas_tocadas": int(len(tocadas)),
        "poblacion_base_malla": round(float(malla_m["POBTOT"].sum()), 2),
        "poblacion_total_estatal_censo_2020": POBLACION_ESTATAL_CENSO_2020,
        "poblacion_estimada_area": round(poblacion_prorrateada, 2),
        "porcentaje_sobre_poblacion_estatal": round(
            100 * poblacion_prorrateada / POBLACION_ESTATAL_CENSO_2020, 6
        ),
        "poblacion_celdas_completas_cota_alta": round(float(tocadas["POBTOT"].sum()), 2),
    }
    return tocadas.to_crs("EPSG:4326"), resumen


def capa_ligera_para_mapa(capa, campos):
    """Deja solo los campos del tooltip y redondea las coordenadas (unos 10 cm)."""
    ligera = capa[[*campos, "geometry"]].to_crs("EPSG:4326").copy()
    ligera["geometry"] = shapely.set_precision(ligera.geometry.array, 1e-6)
    return ligera


In [16]:
malla = cargar_malla(MALLA_INEGI_DIR / "Datos totales.csv")
celdas_estacion = cargar_malla(MALLA_INEGI_DIR / "Datos filtrados.csv")
# REL_H_M son hombres por cada 100 mujeres; de ahí se reparte la población total.
malla["hombres_estimados"] = malla["POBTOT"] * malla["REL_H_M"] / (100 + malla["REL_H_M"])
malla["mujeres_estimadas"] = malla["POBTOT"] - malla["hombres_estimados"]
malla_m = malla.to_crs(CRS_METRICO_QUERETARO)

# Demanda general: paradas con acceso confiable dentro del umbral principal.
celdas_cubiertas, resumen_demanda = estimar_demanda(
    malla_m, paradas_mapa_30, RADIO_DEMANDA_M
)

# Escenarios acumulados de acceso confiable por P80.
umbrales_minutos = [5, 15, 30, 45, 60]
sufijo_umbrales = "_".join(map(str, umbrales_minutos))
# GeoPackage con una capa de paradas y otra de celdas por escenario, para QGIS.
QGIS_PATH = PROCESSED_DIR / "acceso_corregidora_qgis.gpkg"
QGIS_PATH.unlink(missing_ok=True)
resumen_umbrales = []
celdas_por_umbral = {}
paradas_umbral_csv = []
filas_tabla_acceso = []
cobertura_indicadores = []
for umbral_min in umbrales_minutos:
    paradas_umbral = resultados[
        resultados["stop_id"].ne(corregidora_stop_id)
        & resultados["observaciones"].ge(MIN_OBSERVACIONES_HISTORICAS)
        & resultados["p80_historico_min"].le(umbral_min)
    ].copy()
    paradas_umbral["umbral_min"] = umbral_min
    paradas_umbral_csv.append(paradas_umbral)
    celdas_umbral, _ = estimar_demanda(malla_m, paradas_umbral, RADIO_DEMANDA_M)
    celdas_umbral = celdas_umbral.assign(umbral_min=umbral_min)
    celdas_por_umbral[umbral_min] = celdas_umbral
    resumen_umbrales.append({
        "umbral_min": umbral_min,
        "paradas": len(paradas_umbral),
        "celdas": len(celdas_umbral),
        "poblacion_prorrateada": round(float(celdas_umbral["POBTOT_prorrateada"].sum()), 2),
        "poblacion_celdas_completas": round(float(celdas_umbral["POBTOT"].sum()), 2),
    })
    filas_tabla_acceso.append({
        "Tiempo": f"{umbral_min} min",
        "Paradas": len(paradas_umbral),
        "Celdas tocadas": len(celdas_umbral),
        "Personas (celdas completas)": celdas_umbral["POBTOT"].sum(),
        **{
            etiqueta: celdas_umbral[f"{columna}_prorrateada"].sum(min_count=1)
            for columna, etiqueta in INDICADORES_MALLA.items()
        },
    })
    cobertura_indicadores.append({
        "tiempo": umbral_min,
        "celdas": len(celdas_umbral),
        **{
            f"con_dato_{columna}": int(celdas_umbral[columna].notna().sum())
            for columna in INDICADORES_MALLA
        },
    })
    if celdas_umbral.empty:
        continue
    tabla_con_coordenadas(celdas_umbral).to_csv(
        PROCESSED_DIR / f"celdas_acceso_{umbral_min}_min.csv",
        index=False, encoding="utf-8-sig",
    )
    puntos_umbral = gpd.GeoDataFrame(
        paradas_umbral,
        geometry=gpd.points_from_xy(paradas_umbral["stop_lon"], paradas_umbral["stop_lat"]),
        crs="EPSG:4326",
    )
    guardar_capa_qgis(celdas_umbral, QGIS_PATH, f"celdas_acceso_{umbral_min}_min")
    guardar_capa_qgis(puntos_umbral, QGIS_PATH, f"paradas_acceso_{umbral_min}_min")

resumen_umbrales = pd.DataFrame(resumen_umbrales)
# X (longitud) e Y (latitud) en EPSG:4326: QGIS las detecta al importar el CSV.
paradas_umbrales = pd.concat(paradas_umbral_csv, ignore_index=True).assign(
    X=lambda tabla: tabla["stop_lon"], Y=lambda tabla: tabla["stop_lat"]
)
# Además del archivo completo (5 a 60 min) se regeneran los de 5-30 y 5-45 min,
# para que ningún archivo de escenarios quede con un buffer o un histórico anterior.
for cantidad in range(3, len(umbrales_minutos) + 1):
    incluidos = umbrales_minutos[:cantidad]
    sufijo = "_".join(map(str, incluidos))
    resumen_umbrales[resumen_umbrales["umbral_min"].isin(incluidos)].to_csv(
        PROCESSED_DIR / f"resumen_acceso_{sufijo}_min.csv", index=False, encoding="utf-8-sig"
    )
    paradas_umbrales[paradas_umbrales["umbral_min"].isin(incluidos)].to_csv(
        PROCESSED_DIR / f"paradas_acceso_{sufijo}_min.csv", index=False, encoding="utf-8-sig"
    )
tabla_personas_acceso = pd.DataFrame(filas_tabla_acceso)
tabla_personas_acceso.to_csv(
    PROCESSED_DIR / f"personas_acceso_{sufijo_umbrales}_min.csv",
    index=False, encoding="utf-8-sig",
)

DEMANDA_CSV_PATH = PROCESSED_DIR / "demanda_potencial_celdas.csv"
DEMANDA_RESUMEN_PATH = PROCESSED_DIR / "demanda_potencial_resumen.json"
tabla_con_coordenadas(celdas_cubiertas).to_csv(
    DEMANDA_CSV_PATH, index=False, encoding="utf-8-sig"
)
paradas_demanda = gpd.GeoDataFrame(
    paradas_mapa_30.copy(),
    geometry=gpd.points_from_xy(paradas_mapa_30["stop_lon"], paradas_mapa_30["stop_lat"]),
    crs="EPSG:4326",
)
PARADAS_DEMANDA_CSV_PATH = PROCESSED_DIR / "paradas_demanda_potencial.csv"
tabla_con_coordenadas(paradas_demanda).to_csv(
    PARADAS_DEMANDA_CSV_PATH, index=False, encoding="utf-8-sig"
)
DEMANDA_RESUMEN_PATH.write_text(
    json.dumps(resumen_demanda, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Malla: {len(malla):,} celdas de nivel 9 (unos 468 m por lado).")
print(f"Capas para QGIS: {QGIS_PATH.resolve()}")
print(f"Celdas exportadas: {DEMANDA_CSV_PATH.resolve()}")
print(f"Paradas exportadas: {PARADAS_DEMANDA_CSV_PATH.resolve()}")
display(pd.DataFrame.from_dict(resumen_demanda, orient="index", columns=["valor"]))

# Validación de la geometría reconstruida: las seis celdas de "Datos filtrados"
# deben rodear la estación.
estacion_m = (
    gpd.GeoSeries(gpd.points_from_xy([CORREGIDORA_LON], [CORREGIDORA_LAT]), crs="EPSG:4326")
    .to_crs(CRS_METRICO_QUERETARO)
    .iloc[0]
)
celdas_estacion_m = celdas_estacion.to_crs(CRS_METRICO_QUERETARO)
assert celdas_estacion_m.union_all().contains(estacion_m)
print(
    f"Datos filtrados: {len(celdas_estacion)} celdas alrededor de la estación con "
    f"{celdas_estacion['POBTOT'].sum():,.0f} personas; la estación queda dentro del bloque."
)

mapa_demanda = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON], zoom_start=13, tiles="OpenStreetMap"
)
campos_tooltip_celda = ["CODIGO", "POBTOT", "fraccion_area_cubierta", "POBTOT_prorrateada"]
alias_tooltip_celda = [
    "Celda:", "Población de la celda:",
    "Fracción dentro de buffers:", "Población prorrateada:",
]
capa_celdas_general = folium.GeoJson(
    capa_ligera_para_mapa(celdas_cubiertas, campos_tooltip_celda),
    name=f"Celdas (demanda general, P80 ≤ {UMBRAL_ANALISIS_MIN:g} min)", show=False,
    style_function=lambda _: {
        "color": "#7e22ce", "weight": 1,
        "fillColor": "#c084fc", "fillOpacity": 0.32,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=campos_tooltip_celda, aliases=alias_tooltip_celda, localize=True
    ),
).add_to(mapa_demanda)
# La celda siguiente busca en estas capas la celda del punto seleccionado.
capas_celdas_mapa = [(-1, capa_celdas_general)]
# Capas de acceso acumuladas; activa/desactiva cada umbral desde LayerControl.
colores_acceso = {5: "#22c55e", 15: "#f59e0b", 30: "#ef4444", 45: "#8b5cf6", 60: "#0ea5e9"}
for umbral_min in umbrales_minutos:
    celdas_umbral = celdas_por_umbral[umbral_min]
    if celdas_umbral.empty:
        continue
    color_acceso = colores_acceso[umbral_min]
    capa_acceso = folium.GeoJson(
        capa_ligera_para_mapa(celdas_umbral, campos_tooltip_celda),
        name=f"Acceso ≤{umbral_min} min ({len(celdas_umbral)} celdas)",
        show=(umbral_min == 5),
        style_function=lambda _, color=color_acceso: {
            "color": color, "weight": 1,
            "fillColor": color, "fillOpacity": 0.3,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=campos_tooltip_celda, aliases=alias_tooltip_celda, localize=True
        ),
    ).add_to(mapa_demanda)
    capas_celdas_mapa.append((umbral_min, capa_acceso))
folium.GeoJson(
    celdas_estacion[["CODIGO", "POBTOT", "geometry"]],
    name="Celdas alrededor de la estación (Datos filtrados)", show=False,
    style_function=lambda _: {"color": "#111827", "weight": 3, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(
        fields=["CODIGO", "POBTOT"], aliases=["Celda:", "Población:"], localize=True
    ),
).add_to(mapa_demanda)

capa_rutas_demanda = folium.FeatureGroup(name="Caminos hacia Corregidora", show=True)
capa_buffers = folium.FeatureGroup(
    name=f"Buffers de {2 * RADIO_DEMANDA_M:g} m de diámetro", show=True
)
agregar_segmentos_rutas(capa_rutas_demanda, segmentos_mapa_30, opacity=0.55)
for fila in paradas_mapa_30.itertuples(index=False):
    route_id_inicio = ruta_inicial(fila.rutas_por_segmento)
    color_ruta = color_por_ruta(route_id_inicio)
    folium.Circle(
        [fila.stop_lat, fila.stop_lon], radius=RADIO_DEMANDA_M,
        color=color_ruta, weight=1, fill=True, fill_color=color_ruta,
        fill_opacity=0.12,
    ).add_to(capa_buffers)
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon], radius=3, color="#111827",
        weight=1, fill=True, fill_color=color_ruta, fill_opacity=0.95,
        tooltip=f"{fila.stop_name} | {fila.itinerario_rutas}",
    ).add_to(capa_buffers)

folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON], tooltip="Estación Corregidora",
    icon=folium.Icon(color="red", icon="train", prefix="fa"),
).add_to(mapa_demanda)
capa_rutas_demanda.add_to(mapa_demanda)
capa_buffers.add_to(mapa_demanda)
agregar_leyenda_rutas(mapa_demanda, segmentos_mapa_30, "Rutas y demanda potencial")
agregar_control_capas(mapa_demanda)
mapa_demanda.fit_bounds([
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    *paradas_mapa_30[["stop_lat", "stop_lon"]].to_numpy().tolist(),
])
# El mapa se vuelve interactivo y se guarda en la celda siguiente.


Malla: 38,937 celdas de nivel 9 (unos 468 m por lado).
Capas para QGIS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/acceso_corregidora_qgis.gpkg
Celdas exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/demanda_potencial_celdas.csv
Paradas exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/paradas_demanda_potencial.csv


,valor
fuente,"Malla geoestadística del INEGI, nivel 9, con i..."
metodo,prorrateo por área de las celdas de la malla t...
radio_buffer_m,400.0
numero_paradas,327
numero_celdas_malla,38937
numero_celdas_tocadas,249
poblacion_base_malla,7042690.0
poblacion_total_estatal_censo_2020,2368467
poblacion_estimada_area,227442.07
porcentaje_sobre_poblacion_estatal,9.602923


Datos filtrados: 6 celdas alrededor de la estación con 11,548 personas; la estación queda dentro del bloque.


### 13.1 Mapa interactivo: ruta óptima desde cualquier punto

Al hacer clic en cualquier punto del mapa de demanda, un panel muestra:

- **Ruta óptima hacia Corregidora:** entre las paradas a `RADIO_BUSQUEDA_PARADA_M` metros o menos, se elige la que minimiza *caminata + P80 histórico* (acceso confiable). La caminata se aproxima como línea recta × `FACTOR_DESVIO_CAMINATA` a 80 m/min. Si ninguna parada cercana tiene muestra suficiente, se usa el tiempo programado GTFS y el panel lo advierte. El mapa dibuja la caminata (gris punteado), cada tramo de autobús con el color de su ruta y los puntos de subida y transbordo; el panel lista los pasos y hasta tres alternativas.
- **Datos de la celda de la malla** donde cae el punto (Censo 2020): población, mujeres y hombres estimados, niños de 0 a 14 años, mayores de 60, personas con discapacidad, población económicamente activa, viviendas habitadas y el escenario P80 más corto que la toca, con la fracción de la celda dentro de los buffers y las personas que aporta por prorrateo. Solo hay datos para celdas tocadas por algún buffer; funciona aunque la capa de celdas esté apagada.

El tiempo de transporte es la ETA histórica de Google desde la parada, y el trazo es el camino mínimo GTFS con transbordos, igual que en los demás mapas. Todo el cálculo ocurre en el navegador con los datos incrustados en el HTML; no requiere servidor ni llamadas externas.

In [17]:
from branca.element import MacroElement
from jinja2 import Template

# Parámetros de la búsqueda desde el punto seleccionado.
RADIO_BUSQUEDA_PARADA_M = 1000  # paradas candidatas alrededor del punto
VELOCIDAD_CAMINATA_M_MIN = 80   # la misma velocidad del escenario sin cruces
FACTOR_DESVIO_CAMINATA = 1.3    # convierte la línea recta en una distancia aproximada por calles


class ScriptMapa(MacroElement):
    """Inserta JavaScript propio después de las capas y controles del mapa."""

    _template = Template("{% macro script(this, kwargs) %}{{ this.codigo }}{% endmacro %}")

    def __init__(self, codigo):
        super().__init__()
        self._name = "ScriptMapa"
        self.codigo = codigo


def json_para_script(datos):
    """Serializa datos para incrustarlos de forma segura dentro de <script>."""
    texto = json.dumps(datos, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    return texto.replace("</", "<\\/")


def numero_o_nulo(valor, decimales=1):
    """Convierte NaN en None para que el JSON sea válido."""
    return None if pd.isna(valor) else round(float(valor), decimales)


def datos_ruta_optima(resultados, stops, trazos, route_name_map, destino_stop_id):
    """Caminos mínimos de todas las paradas, con su trazo por calles y P50/P80."""
    origenes = resultados[resultados["stop_id"].ne(destino_stop_id)]
    paradas_usadas = {destino_stop_id}
    candidatas, rutas_usadas, segmentos = [], set(), {}
    for fila in origenes.itertuples(index=False):
        camino = [str(s) for s in fila.camino_stop_ids]
        rutas = [str(r) for r in fila.rutas_por_segmento]
        if len(camino) < 2 or len(camino) != len(rutas) + 1:
            continue
        paradas_usadas.update(camino)
        rutas_usadas.update(rutas)
        for par in zip(camino, camino[1:]):
            if par in trazos:
                segmentos.setdefault(
                    "|".join(par),
                    [[round(lat, 5), round(lon, 5)] for lat, lon in trazos[par]["coordenadas"]],
                )
        candidatas.append([
            str(fila.stop_id),
            numero_o_nulo(fila.tiempo_red_min),
            numero_o_nulo(fila.p50_historico_min),
            numero_o_nulo(fila.p80_historico_min),
            int(fila.observaciones),
            camino,
            rutas,
        ])
    info_paradas = stops[stops["stop_id"].astype(str).isin(paradas_usadas)]
    return {
        "paradas": {
            str(f.stop_id): [str(f.stop_name), round(f.stop_lat, 6), round(f.stop_lon, 6)]
            for f in info_paradas.itertuples(index=False)
        },
        "candidatas": candidatas,
        "segmentos": segmentos,
        "rutas": {
            r: [etiqueta_ruta(r, route_name_map), color_por_ruta(r), r in RUTAS_CAMINATA]
            for r in sorted(rutas_usadas)
        },
        "destino": str(destino_stop_id),
    }


CAMPOS_CELDA_JS = [
    "POBTOT", "mujeres_estimadas", "hombres_estimados", "POB0_14",
    "P_60YMAS", "PCON_DISC", "PEA", "TVIVHAB",
]


def datos_celdas_js(celdas_por_umbral, celdas_cubiertas):
    """Indicadores por celda de la malla y el escenario P80 más corto que la toca.

    Además de los valores de la celda completa se guarda, para ese escenario,
    la fracción de la celda dentro de los buffers y la población prorrateada.
    """
    escenario_minimo = {}
    for umbral_min in sorted(celdas_por_umbral, reverse=True):
        for fila in celdas_por_umbral[umbral_min].itertuples(index=False):
            escenario_minimo[str(fila.CODIGO)] = [
                umbral_min,
                numero_o_nulo(fila.fraccion_area_cubierta, 3),
                numero_o_nulo(fila.POBTOT_prorrateada, 0),
            ]
    todas = pd.concat(
        [celdas_cubiertas, *celdas_por_umbral.values()], ignore_index=True
    ).drop_duplicates("CODIGO")
    datos = {}
    for fila in todas.itertuples(index=False):
        valores = [numero_o_nulo(getattr(fila, campo), 0) for campo in CAMPOS_CELDA_JS]
        codigo = str(fila.CODIGO)
        datos[codigo] = [*valores, *escenario_minimo.get(codigo, [None, None, None])]
    return datos


JS_RUTA_OPTIMA = """
(function () {
  const mapa = __MAPA__;
  const R = __RUTAS__;
  const M = __CELDAS__;
  const capasCeldas = __CAPAS_CELDAS__;
  const P = __PARAMETROS__;
  const capaSeleccion = L.layerGroup().addTo(mapa);
  const panel = document.getElementById("panel-ruta-optima");
  const cuerpo = panel.querySelector(".cuerpo");
  const esc = (t) => String(t).replace(/[&<>"']/g, (c) => ({
    "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"
  })[c]);
  const fmt = (v, d = 0) => v == null ? "<span style='color:#9ca3af'>reservado</span>"
    : Number(v).toLocaleString("es-MX", { maximumFractionDigits: d, minimumFractionDigits: d });
  const rad = (g) => g * Math.PI / 180;
  const distancia = (a, b) => {
    const dLat = rad(b[0] - a[0]), dLon = rad(b[1] - a[1]);
    const h = Math.sin(dLat / 2) ** 2 +
      Math.cos(rad(a[0])) * Math.cos(rad(b[0])) * Math.sin(dLon / 2) ** 2;
    return 6371008.8 * 2 * Math.atan2(Math.sqrt(h), Math.sqrt(1 - h));
  };
  const chip = (r) => {
    const info = R.rutas[r] || [r, "#6b7280"];
    return "<span style='display:inline-block;padding:0 4px;border-left:8px solid " +
      info[1] + ";background:#f3f4f6;border-radius:2px;font-weight:600'>" + esc(info[0]) + "</span>";
  };
  const nombre = (s) => (R.paradas[s] || [s])[0];
  const coord = (s) => [R.paradas[s][1], R.paradas[s][2]];

  // Punto dentro de un polígono de Leaflet (incluye huecos y multipolígonos).
  const dentroAnillo = (p, anillo) => {
    let dentro = false;
    for (let i = 0, j = anillo.length - 1; i < anillo.length; j = i++) {
      const a = anillo[i], b = anillo[j];
      if ((a.lat > p.lat) !== (b.lat > p.lat) &&
          p.lng < (b.lng - a.lng) * (p.lat - a.lat) / (b.lat - a.lat) + a.lng) dentro = !dentro;
    }
    return dentro;
  };
  const dentroPoligono = (p, latlngs) => {
    if (!latlngs.length) return false;
    if (latlngs[0] instanceof L.LatLng) return dentroAnillo(p, latlngs);
    if (latlngs[0][0] instanceof L.LatLng) {
      return dentroAnillo(p, latlngs[0]) && !latlngs.slice(1).some((h) => dentroAnillo(p, h));
    }
    return latlngs.some((parte) => dentroPoligono(p, parte));
  };
  function buscarCelda(latlng) {
    for (const capa of capasCeldas) {
      let encontrada = null;
      capa.eachLayer((f) => {
        if (!encontrada && f.getBounds && f.getBounds().contains(latlng) &&
            dentroPoligono(latlng, f.getLatLngs())) encontrada = f;
      });
      if (encontrada) return encontrada;
    }
    return null;
  }

  // Ordena las paradas cercanas por caminata + P80 histórico. Si ninguna tiene
  // muestra suficiente, usa el tiempo programado GTFS y lo indica.
  function opciones(latlng) {
    const punto = [latlng.lat, latlng.lng];
    const todas = R.candidatas.map((c) => {
      const recta = distancia(punto, coord(c[0]));
      const caminataM = recta * P.desvio;
      const confiable = c[4] >= P.min_obs && c[3] != null;
      const transporte = confiable ? c[3] : c[1];
      return { c, recta, caminataM, caminataMin: caminataM / P.velocidad, confiable,
               total: caminataM / P.velocidad + transporte };
    });
    let cercanas = todas.filter((o) => o.recta <= P.radio);
    let aviso = "";
    if (!cercanas.length) {
      cercanas = todas.sort((a, b) => a.recta - b.recta).slice(0, 5);
      aviso = "No hay paradas a menos de " + P.radio + " m; se usan las 5 más cercanas.";
    }
    const conHistorico = cercanas.filter((o) => o.confiable);
    const grupo = conHistorico.length ? conHistorico : cercanas;
    if (!conHistorico.length) {
      aviso += (aviso ? " " : "") + "Ninguna parada cercana tiene histórico suficiente: " +
        "el tiempo es el programado GTFS, sin espera ni atrasos.";
    }
    grupo.sort((a, b) => a.total - b.total);
    return { lista: grupo.slice(0, 4), aviso };
  }

  function tramos(camino, rutas) {
    const lista = [];
    rutas.forEach((r, i) => {
      const ultimo = lista[lista.length - 1];
      if (ultimo && ultimo.ruta === r) { ultimo.fin = i + 1; }
      else { lista.push({ ruta: r, inicio: i, fin: i + 1 }); }
    });
    return lista;
  }

  function dibujar(latlng, opcion) {
    capaSeleccion.clearLayers();
    const [stopId, , , , , camino, rutas] = opcion.c;
    L.circleMarker(latlng, { radius: 7, color: "#111827", weight: 2, fillColor: "#facc15",
      fillOpacity: 1 }).bindTooltip("Punto seleccionado").addTo(capaSeleccion);
    L.polyline([latlng, coord(stopId)], { color: "#374151", weight: 4, dashArray: "6 6" })
      .bindTooltip("Caminata ~" + Math.round(opcion.caminataM) + " m").addTo(capaSeleccion);
    const limites = [latlng];
    camino.slice(0, -1).forEach((s, i) => {
      const clave = s + "|" + camino[i + 1];
      const puntos = R.segmentos[clave] || [coord(s), coord(camino[i + 1])];
      const info = R.rutas[rutas[i]] || [rutas[i], "#6b7280", false];
      L.polyline(puntos, { color: "#ffffff", weight: 9, opacity: 0.9 }).addTo(capaSeleccion);
      L.polyline(puntos, { color: info[1], weight: 6, opacity: 1,
        dashArray: info[2] ? "6 6" : null }).bindTooltip("Ruta " + esc(info[0])).addTo(capaSeleccion);
      puntos.forEach((p) => limites.push(p));
    });
    tramos(camino, rutas).forEach((t, i) => {
      const s = camino[t.inicio];
      L.circleMarker(coord(s), { radius: 6, color: "#111827", weight: 2,
        fillColor: i === 0 ? "#22c55e" : "#ffffff", fillOpacity: 1 })
        .bindTooltip((i === 0 ? "Sube: " : "Transbordo: ") + esc(nombre(s))).addTo(capaSeleccion);
    });
    mapa.fitBounds(L.latLngBounds(limites).pad(0.15), { maxZoom: 16 });
  }

  function htmlCelda(celda) {
    if (!celda) {
      return "<div class='seccion'><b>Celda de la malla</b><div style='color:#6b7280'>El " +
        "punto está fuera de las celdas tocadas por los buffers del análisis.</div></div>";
    }
    const clave = String(celda.feature.properties.CODIGO);
    const d = M[clave];
    if (!d) return "<div class='seccion'><b>Celda " + esc(clave) + "</b></div>";
    const filas = [
      ["Población de la celda", fmt(d[0])],
      ["Mujeres (estimadas)", fmt(d[1])], ["Hombres (estimados)", fmt(d[2])],
      ["Niños (0-14 años)", fmt(d[3])], ["Mayores (60 años o más)", fmt(d[4])],
      ["Personas con discapacidad", fmt(d[5])], ["Población económicamente activa", fmt(d[6])],
      ["Viviendas habitadas", fmt(d[7])],
    ];
    const escenario = d[8] == null ? "<div class='nota'>Fuera de los escenarios P80.</div>"
      : "<div style='margin-top:4px'>Escenario de acceso más corto: <b>≤ " + d[8] +
        " min (P80)</b>; " + fmt(100 * d[9]) + "% de la celda queda dentro de los buffers " +
        "(" + fmt(d[10]) + " personas contabilizadas por prorrateo).</div>";
    return "<div class='seccion'><b>Celda " + esc(clave) + "</b> <span class='nota'>(~468 m por lado)</span>" +
      "<table>" + filas.map((f) => "<tr><td>" + f[0] + "</td><td>" + f[1] + "</td></tr>").join("") +
      "</table>" + escenario + "<div class='nota'>Malla geoestadística del INEGI, Censo 2020, " +
      "proporcionada por el Instituto de Movilidad. \\"reservado\\": el INEGI oculta el dato " +
      "en celdas con muy poca población. Mujeres y hombres se estiman con la relación " +
      "hombres-mujeres de la celda.</div></div>";
  }

  function htmlRuta(opcion, otras, aviso, latlng, optima) {
    const [stopId, gtfs, p50, p80, n, camino, rutas] = opcion.c;
    const pasos = ["<li>Camina ~" + fmt(opcion.caminataM) + " m (~" +
      fmt(Math.max(1, opcion.caminataMin)) + " min) a <b>" + esc(nombre(stopId)) + "</b>.</li>"];
    const lista = tramos(camino, rutas);
    lista.forEach((t, i) => {
      const sube = camino[t.inicio], baja = camino[t.fin];
      const info = R.rutas[t.ruta] || [t.ruta, "#6b7280", false];
      const paradas = t.fin - t.inicio;
      if (info[2]) {
        pasos.push("<li>Camina de <b>" + esc(nombre(sube)) + "</b> a <b>" + esc(nombre(baja)) + "</b>.</li>");
      } else {
        pasos.push("<li>" + (i === 0 ? "Toma " : "Transborda a ") + chip(t.ruta) + " en <b>" +
          esc(nombre(sube)) + "</b> y baja en <b>" + esc(nombre(baja)) + "</b> (" + paradas +
          (paradas === 1 ? " parada" : " paradas") + ").</li>");
      }
    });
    pasos.push("<li>Llegas a la parada destino, junto a la estación Corregidora.</li>");
    const tiempo = opcion.confiable
      ? "<b>" + fmt(opcion.total) + " min</b> con P80 (confiable) · " +
        fmt(opcion.caminataMin + p50) + " min con P50 (habitual)"
      : "<b>" + fmt(opcion.total) + " min</b> según el horario GTFS (sin histórico suficiente)";
    const detalle = "Caminata " + fmt(opcion.caminataMin) + " min + transporte " +
      (opcion.confiable ? "P80 " + fmt(p80, 1) + " min (n=" + n + ")" : "GTFS " + fmt(gtfs, 1) + " min") +
      " · " + (lista.length - 1) + (lista.length - 1 === 1 ? " transbordo" : " transbordos");
    let h = "<div class='seccion'><b>" + (opcion === optima ? "Ruta óptima a Corregidora"
      : "Alternativa a Corregidora") + "</b>" +
      (aviso ? "<div class='aviso'>" + esc(aviso) + "</div>" : "") +
      "<div style='margin:4px 0'>" + tiempo + "</div><div class='nota'>" + detalle + "</div>" +
      "<ol>" + pasos.join("") + "</ol>";
    const directo = distancia([latlng.lat, latlng.lng], P.estacion) * P.desvio;
    const minutosDirecto = directo / P.velocidad;
    if (minutosDirecto <= opcion.total) {
      h += "<div class='aviso'>Caminar directo a la estación (~" + fmt(directo) + " m, ~" +
        fmt(minutosDirecto) + " min) es igual o más rápido.</div>";
    }
    if (otras.length) {
      h += "<div style='margin-top:6px'><b>" + (opcion === optima ? "Alternativas" : "Otras opciones") +
        "</b></div><ul class='alternativas'>" +
        otras.map((o, i) => "<li><a href='#' data-i='" + (i + 1) + "'>" + esc(nombre(o.c[0])) +
          "</a>" + (o === optima ? " <b>(óptima)</b>" : "") + " · " + fmt(o.total) + " min · " + o.c[6].filter((r, k, a) => r !== a[k - 1])
          .map((r) => chip(r)).join(" → ") + "</li>").join("") + "</ul>";
    }
    return h + "<div class='nota'>Tiempo de transporte: ETA histórica de Google desde la parada. " +
      "Trazo: camino mínimo GTFS con transbordos. Caminata: línea recta × " + P.desvio +
      " a " + P.velocidad + " m/min.</div></div>";
  }

  function seleccionar(latlng, indice = 0) {
    const { lista, aviso } = opciones(latlng);
    const celda = buscarCelda(latlng);
    if (!lista.length) {
      cuerpo.innerHTML = htmlCelda(celda) + "<div class='seccion'>Sin paradas disponibles.</div>";
      panel.style.display = "block";
      return;
    }
    const elegida = lista[indice];
    dibujar(latlng, elegida);
    if (celda) {
      L.geoJSON(celda.toGeoJSON(), { style: { color: "#ca8a04", weight: 3, fillColor: "#facc15",
        fillOpacity: 0.25 }, interactive: false }).addTo(capaSeleccion);
    }
    const otras = lista.filter((_, i) => i !== indice);
    cuerpo.innerHTML = htmlRuta(elegida, otras, aviso, latlng, lista[0]) + htmlCelda(celda);
    cuerpo.querySelectorAll("a[data-i]").forEach((a) => a.addEventListener("click", (e) => {
      e.preventDefault();
      const destino = lista.indexOf(otras[Number(a.dataset.i) - 1]);
      seleccionar(latlng, destino);
    }));
    panel.style.display = "block";
  }

  mapa.on("click", (e) => seleccionar(e.latlng));
  panel.querySelector(".cerrar").addEventListener("click", () => {
    capaSeleccion.clearLayers();
    cuerpo.innerHTML = panel.dataset.inicio;
  });
  panel.dataset.inicio = cuerpo.innerHTML;
  mapa.getContainer().style.cursor = "crosshair";
})();
"""

PANEL_RUTA_OPTIMA = """
<style>
#panel-ruta-optima { position: fixed; top: 12px; left: 56px; z-index: 9999; width: 330px;
  max-height: calc(100vh - 210px); overflow-y: auto; background: #fff; border: 1px solid #777;
  border-radius: 5px; box-shadow: 0 1px 5px #999; font: 12px/1.35 sans-serif; color: #111827; }
#panel-ruta-optima .titulo { display: flex; justify-content: space-between; align-items: center;
  padding: 6px 8px; background: #f3f4f6; border-bottom: 1px solid #ddd; font-weight: bold;
  position: sticky; top: 0; }
#panel-ruta-optima .cerrar { cursor: pointer; border: 0; background: none; font-size: 14px; }
#panel-ruta-optima .cuerpo { padding: 4px 8px 8px; }
#panel-ruta-optima .seccion { margin-top: 6px; }
#panel-ruta-optima ol, #panel-ruta-optima ul { margin: 4px 0; padding-left: 18px; }
#panel-ruta-optima li { margin: 3px 0; }
#panel-ruta-optima table { width: 100%; border-collapse: collapse; margin-top: 3px; }
#panel-ruta-optima td { padding: 1px 0; border-bottom: 1px solid #f3f4f6; }
#panel-ruta-optima td:last-child { text-align: right; font-variant-numeric: tabular-nums; }
#panel-ruta-optima .nota { color: #6b7280; font-size: 11px; margin-top: 3px; }
#panel-ruta-optima .aviso { background: #fef3c7; border-left: 3px solid #f59e0b;
  padding: 3px 6px; margin: 4px 0; }
@media (max-width: 600px) { #panel-ruta-optima { left: 8px; right: 8px; width: auto;
  top: auto; bottom: 8px; max-height: 45vh; } }
</style>
<div id="panel-ruta-optima">
  <div class="titulo"><span>Ruta óptima y datos de la celda</span>
    <button class="cerrar" title="Limpiar selección">✕</button></div>
  <div class="cuerpo"><div class="seccion">Haz clic en cualquier punto del mapa para ver
    la ruta óptima hacia la estación Corregidora y los datos censales de la celda de la malla.</div></div>
</div>
"""

datos_rutas_js = datos_ruta_optima(
    resultados, stops, trazos_red_vial, route_name_map, corregidora_stop_id
)
datos_celdas = datos_celdas_js(celdas_por_umbral, celdas_cubiertas)
parametros_js = {
    "radio": RADIO_BUSQUEDA_PARADA_M,
    "velocidad": VELOCIDAD_CAMINATA_M_MIN,
    "desvio": FACTOR_DESVIO_CAMINATA,
    "min_obs": MIN_OBSERVACIONES_HISTORICAS,
    "estacion": [CORREGIDORA_LAT, CORREGIDORA_LON],
}
# La capa más grande va primero: contiene a las demás y encuentra la celda antes.
capas_busqueda = sorted(capas_celdas_mapa, key=lambda par: -par[0])
codigo_js = (
    JS_RUTA_OPTIMA.replace("__MAPA__", mapa_demanda.get_name())
    .replace("__RUTAS__", json_para_script(datos_rutas_js))
    .replace("__CELDAS__", json_para_script(datos_celdas))
    .replace("__CAPAS_CELDAS__", "[" + ",".join(c.get_name() for _, c in capas_busqueda) + "]")
    .replace("__PARAMETROS__", json_para_script(parametros_js))
)
mapa_demanda.get_root().html.add_child(folium.Element(PANEL_RUTA_OPTIMA))
mapa_demanda.add_child(ScriptMapa(codigo_js))

print(f"Paradas candidatas para la ruta óptima: {len(datos_rutas_js['candidatas']):,}")
print(f"Celdas con datos al hacer clic: {len(datos_celdas):,}")

MAPA_DEMANDA_PATH = MAPS_DIR / "mapa_demanda_potencial.html"
mapa_demanda.save(MAPA_DEMANDA_PATH)
print(f"Copia HTML: {MAPA_DEMANDA_PATH.resolve()}")
# La última expresión también integra el mapa directamente en el notebook.
mapa_demanda


Paradas candidatas para la ruta óptima: 2,258
Celdas con datos al hacer clic: 662


Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_demanda_potencial.html


### 13.2 Personas con acceso confiable mediante QroBus

La siguiente tabla resume la estimación principal. Se usa la unión espacial de todos los buffers, por lo que una persona cubierta por varias paradas no se cuenta varias veces.

La tabla acumulada por tiempo de acceso (5, 15, 30, 45 y 60 minutos de P80) prorratea por área cada indicador de la malla: personas, mujeres y hombres estimados, niños de 0 a 14 años (`POB0_14`), mayores de 60 (`P_60YMAS`), personas con discapacidad (`PCON_DISC`), población económicamente activa (`PEA`) y viviendas habitadas (`TVIVHAB`). La columna de celdas completas es el techo. Las celdas con datos reservados no suman a ese indicador, por lo que los grupos son un mínimo; la tabla de cobertura indica cuántas celdas tienen cada dato.

In [18]:
tabla_acceso_qrobus = pd.DataFrame(
    [
        {
            "escenario": "Acceso confiable mediante QroBus",
            "criterio": (
                f"P80 histórico <= {UMBRAL_ANALISIS_MIN:g} min; "
                f"n >= {MIN_OBSERVACIONES_HISTORICAS}"
            ),
            "paradas_conectadas": int(resumen_demanda["numero_paradas"]),
            "radio_acceso_m": float(resumen_demanda["radio_buffer_m"]),
            "personas_con_acceso_aprox": int(
                round(resumen_demanda["poblacion_estimada_area"])
            ),
            "personas_celdas_completas_techo": int(
                round(resumen_demanda["poblacion_celdas_completas_cota_alta"])
            ),
            "porcentaje_poblacion_estatal": float(
                resumen_demanda["porcentaje_sobre_poblacion_estatal"]
            ),
            "fuente": resumen_demanda["fuente"],
            "metodo": resumen_demanda["metodo"],
        }
    ]
)
display(
    tabla_acceso_qrobus.style.format(
        {
            "paradas_conectadas": "{:,.0f}",
            "radio_acceso_m": "{:,.0f}",
            "personas_con_acceso_aprox": "{:,.0f}",
            "personas_celdas_completas_techo": "{:,.0f}",
            "porcentaje_poblacion_estatal": "{:.4f}%",
        }
    )
)
print(
    f"Aproximadamente {tabla_acceso_qrobus.iloc[0]['personas_con_acceso_aprox']:,} "
    f"personas tienen acceso confiable mediante QroBus dentro de un radio "
    f"de {RADIO_DEMANDA_M:g} m alrededor de las paradas (prorrateo por área de la malla)."
)
print("Población acumulada por tiempo de acceso (prorrateo por área; celdas completas como techo):")
display(tabla_personas_acceso.style.format(
    {columna: "{:,.0f}" for columna in tabla_personas_acceso.columns if columna != "Tiempo"}
))
print("Cobertura de indicadores (celdas tocadas con dato disponible):")
display(pd.DataFrame(cobertura_indicadores))


,escenario,criterio,paradas_conectadas,radio_acceso_m,personas_con_acceso_aprox,personas_celdas_completas_techo,porcentaje_poblacion_estatal,fuente,metodo
0,Acceso confiable mediante QroBus,P80 histórico <= 30 min; n >= 5,327,400,"227,442","323,435",9.6029%,"Malla geoestadística del INEGI, nivel 9, con indicadores del Censo 2020; proporcionada por el Instituto de Movilidad",prorrateo por área de las celdas de la malla tocadas por los buffers


Aproximadamente 227,442 personas tienen acceso confiable mediante QroBus dentro de un radio de 400 m alrededor de las paradas (prorrateo por área de la malla).
Población acumulada por tiempo de acceso (prorrateo por área; celdas completas como techo):


,Tiempo,Paradas,Celdas tocadas,Personas (celdas completas),Personas,Mujeres (estimadas),Hombres (estimados),Niños (0-14 años),Mayores (60 años o más),Personas con discapacidad,Personas activas económicamente,Viviendas habitadas
0,5 min,1,8,"14,451","3,912","2,068","1,844",526,896,250,"2,236","1,289"
1,15 min,66,54,"62,378","48,158","25,126","23,032","7,012","9,530","2,663","27,470","15,000"
2,30 min,327,249,"323,435","227,442","117,987","109,450","38,838","37,965","10,960","128,970","67,325"
3,45 min,775,529,"656,846","533,057","274,854","258,170","101,946","73,254","22,419","299,490","155,217"
4,60 min,"1,019",662,"817,840","690,982","355,483","335,462","137,752","87,746","28,145","387,104","200,827"


Cobertura de indicadores (celdas tocadas con dato disponible):


,tiempo,celdas,con_dato_POBTOT,con_dato_mujeres_estimadas,con_dato_hombres_estimados,con_dato_POB0_14,con_dato_P_60YMAS,con_dato_PCON_DISC,con_dato_PEA,con_dato_TVIVHAB
0,5,8,8,8,8,8,8,8,8,8
1,15,54,54,54,54,53,54,49,54,54
2,30,249,249,247,247,246,246,236,247,249
3,45,529,529,520,520,517,513,493,520,529
4,60,662,662,652,652,648,642,618,652,662


## 14. Interpretación correcta

El resultado responde: **“¿Cuántas personas viven aproximadamente dentro del buffer de las paradas cuyo P80 histórico indica acceso confiable a Corregidora en 30 minutos o menos?”**

No responde todavía:

- cuál unidad específica llegará;
- si un transbordo concreto estará sincronizado;
- el atraso GPS real del autobús;
- qué servicio opera en una fecha particular, porque el conjunto local no incluye `calendar.txt` ni `calendar_dates.txt`.

Para operación en tiempo real debe incorporarse GTFS-Realtime `TripUpdates` y, de ser posible, `VehiclePositions`.
